
# CELDA 1 — Configuración general del HBM v2 para no2.5

# **Objetivo:**
dejar definidas las rutas, parámetros y nombres de columnas que usará
el HBM v2 con observaciones reales de estaciones.

# **Qué hace esta celda:**
 1. Define la carpeta de trabajo y las rutas de entrada.
 2. Declara el contaminante objetivo (`NO2`).
 3. Define las covariables base del HBM:
   - `Vel_viento_idw`
   - `diff_NO2_grid`
   - `grad_NO2_grid`
    - `adv_proxy_NO2_grid`
 4. Crea la carpeta de salida donde se guardarán paneles, folds,
    resultados de validación y superficies finales.

 **Nota metodológica importante:**
 En esta versión no se usa `NO2_idw` como predictor directo del HBM.
 La respuesta será la observación real de estación (`NO2_obs`).


In [1]:
# %%
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# =========================
# 0) Rutas
# =========================
PROJECT_ROOT = Path.cwd()

# Ajusta esta línea si tus archivos están en otra carpeta
DATA_DIR = PROJECT_ROOT

OBS_CSV  = DATA_DIR / "panel_ambiental_mensual_2020_2024_coords_corregidas.csv"
GRID_CSV = DATA_DIR / "grid_3km_AD_mensual_2020_2024.csv"
W_CSV    = DATA_DIR / "W_grid_3km_queen.csv"

OUT_DIR = PROJECT_ROOT / "HBM_NO2_V2_OUT"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# 1) Configuración HBM v2
# =========================
POLL = "NO2"  
OBS_COL = "NO2"
VALID_COL = "valido_NO2"

X_COLS_RAW = [
    "Vel_viento_idw",
    "diff_NO2_grid",
    "grad_NO2_grid",
    "adv_proxy_NO2_grid",
]

USE_LOG = True
EPS = 1e-6

# folds temporales
FOLDS = {
    "fold_1": {"train_years": [2020, 2021], "test_years": [2022]},
    "fold_2": {"train_years": [2020, 2021, 2022], "test_years": [2023]},
    "fold_3": {"train_years": [2020, 2021, 2022, 2023], "test_years": [2024]},
}

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR     :", DATA_DIR)
print("OUT_DIR      :", OUT_DIR)
print("\nArchivos de entrada:")
for p in [OBS_CSV, GRID_CSV, W_CSV]:
    print(" -", p.name, "| exists:", p.exists())

for p in [OBS_CSV, GRID_CSV, W_CSV]:
    if not p.exists():
        raise FileNotFoundError(f"No encuentro el archivo: {p}")

print("\nConfiguración lista.")
print("POLL         :", POLL)
print("OBS_COL      :", OBS_COL)
print("VALID_COL    :", VALID_COL)
print("X_COLS_RAW   :", X_COLS_RAW)
print("USE_LOG      :", USE_LOG)


PROJECT_ROOT: d:\TRABAJO DE GRADO BEN-MAP\CODIGO
DATA_DIR     : d:\TRABAJO DE GRADO BEN-MAP\CODIGO
OUT_DIR      : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT

Archivos de entrada:
 - panel_ambiental_mensual_2020_2024_coords_corregidas.csv | exists: True
 - grid_3km_AD_mensual_2020_2024.csv | exists: True
 - W_grid_3km_queen.csv | exists: True

Configuración lista.
POLL         : NO2
OBS_COL      : NO2
VALID_COL    : valido_NO2
X_COLS_RAW   : ['Vel_viento_idw', 'diff_NO2_grid', 'grad_NO2_grid', 'adv_proxy_NO2_grid']
USE_LOG      : True



# CELDA 2 — Cargar, limpiar y construir el panel base de modelación

# **Objetivo:**
# construir dos paneles:

 1. `grid_base`:
    la superficie completa celda–mes de la malla 3 km con covariables A–D.

 2. `obs_panel`:
    las observaciones reales de estación para no2.5, ya enlazadas a:
   - `cell_id`
    - `fecha`
    - covariables de la malla

 **Qué hace esta celda:**
 1. Carga el panel de estaciones corregido.
 2. Carga la malla 3 km con términos A–D.
 3. Convierte fechas a formato mensual.
 4. Filtra observaciones válidas de no2.5.
 5. Une cada observación con la covariable de su celda y mes.
 6. Crea índices globales:
    - `cell_idx`
    - `time_idx`
 7. Guarda paneles limpios para el HBM.


In [3]:
# %%
# =========================
# 2) Cargar archivos
# =========================
obs = pd.read_csv(OBS_CSV)
grid = pd.read_csv(GRID_CSV)
w = pd.read_csv(W_CSV)

# =========================
# 3) Fechas
# =========================
obs["fecha"] = pd.to_datetime(
    dict(year=obs["Año"].astype(int), month=obs["Mes"].astype(int), day=1),
    errors="coerce"
)
grid["fecha"] = pd.to_datetime(grid["fecha"], errors="coerce")

if obs["fecha"].isna().any():
    raise ValueError("Hay fechas inválidas en el panel de estaciones.")
if grid["fecha"].isna().any():
    raise ValueError("Hay fechas inválidas en la malla 3km.")

# =========================
# 4) Validación básica de columnas
# =========================
obs_needed = ["Estacion", "cell_id", "fecha", OBS_COL, VALID_COL]
grid_needed = ["cell_id", "fecha"] + X_COLS_RAW

for c in obs_needed:
    if c not in obs.columns:
        raise ValueError(f"Falta la columna '{c}' en el panel de estaciones.")

for c in grid_needed:
    if c not in grid.columns:
        raise ValueError(f"Falta la columna '{c}' en la malla 3km.")

# =========================
# 5) Filtrar observaciones válidas no2.5
# =========================
obs[VALID_COL] = obs[VALID_COL].astype(bool)

obs_no = obs.loc[
    (obs[VALID_COL] == True) &
    (obs[OBS_COL].notna()) &
    (obs["cell_id"].notna())
].copy()

obs_no = obs_no.rename(columns={OBS_COL: "NO2_obs"})

# =========================
# 6) Construir grid_base
# =========================
grid_base = grid[["cell_id", "fecha"] + X_COLS_RAW].copy()
grid_base["year"] = grid_base["fecha"].dt.year
grid_base["month"] = grid_base["fecha"].dt.month

# =========================
# 7) Unir observaciones con covariables de malla
# =========================
obs_panel = obs_no.merge(
    grid_base,
    on=["cell_id", "fecha"],
    how="left",
    validate="many_to_one"
)

missing_merge = obs_panel[X_COLS_RAW].isna().any(axis=1).sum()
if missing_merge > 0:
    raise ValueError(
        f"Hay {missing_merge} observaciones sin covariables de la malla. "
        "Revisa cell_id y fecha."
    )

obs_panel["year"] = obs_panel["fecha"].dt.year
obs_panel["month"] = obs_panel["fecha"].dt.month

# =========================
# 8) Índices globales de celda y tiempo
# =========================
cell_ids = np.sort(grid_base["cell_id"].unique())
time_ids = np.sort(grid_base["fecha"].unique())

cell_map = {cid: i for i, cid in enumerate(cell_ids)}
time_map = {tt: j for j, tt in enumerate(time_ids)}

grid_base["cell_idx"] = grid_base["cell_id"].map(cell_map).astype(int)
grid_base["time_idx"] = grid_base["fecha"].map(time_map).astype(int)

obs_panel["cell_idx"] = obs_panel["cell_id"].map(cell_map).astype(int)
obs_panel["time_idx"] = obs_panel["fecha"].map(time_map).astype(int)

# =========================
# 9) Guardar paneles base
# =========================
grid_base_path = OUT_DIR / "HBM_NO2_grid_base_v2.csv"
obs_panel_path = OUT_DIR / "HBM_NO2_obs_panel_v2.csv"

grid_base.to_csv(grid_base_path, index=False)
obs_panel.to_csv(obs_panel_path, index=False)

# =========================
# 10) Resumen
# =========================
print("Resumen del panel base HBM v2")
print("- obs_panel filas          :", len(obs_panel))
print("- estaciones únicas       :", obs_panel["Estacion"].nunique())
print("- celdas con observación   :", obs_panel["cell_id"].nunique())
print("- grid_base filas          :", len(grid_base))
print("- celdas totales en malla  :", len(cell_ids))
print("- meses totales            :", len(time_ids))
print("\nArchivos guardados:")
print(" -", grid_base_path)
print(" -", obs_panel_path)


Resumen del panel base HBM v2
- obs_panel filas          : 691
- estaciones únicas       : 15
- celdas con observación   : 13
- grid_base filas          : 15240
- celdas totales en malla  : 254
- meses totales            : 60

Archivos guardados:
 - d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_grid_base_v2.csv
 - d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_obs_panel_v2.csv



# # CELDA 3 — Preparar vecindad espacial y folds temporales

# **Objetivo:**
# dejar lista la estructura espacial del HBM y la validación temporal.

# **Qué hace esta celda:**
 1. Toma `W_grid_3km_queen.csv`.
 2. Verifica que las celdas de la vecindad existan en la malla.
 3. Convierte `cell_id` y `neighbor_id` a índices internos `i`, `j`.
 4. Elimina duplicados dirigidos para dejar pares únicos no dirigidos.
 5. Guarda la vecindad final para el modelo.
 6. Guarda también la definición de folds temporales.

# **Salidas:**
# - `HBM_NO2_W_edges_queen_v2.csv`
# - `HBM_NO2_folds_v2.json`

In [4]:
# %%
# =========================
# CELDA 3) Vecindad espacial y folds temporales
# =========================

# Validación mínima de columnas en W
if not {"cell_id", "neighbor_id"}.issubset(w.columns):
    raise ValueError("W_grid_3km_queen.csv debe tener las columnas: 'cell_id' y 'neighbor_id'.")

# Filtrar solo relaciones cuyos nodos existan en la malla
w_use = w[
    w["cell_id"].isin(cell_ids) &
    w["neighbor_id"].isin(cell_ids)
].copy()

if len(w_use) == 0:
    raise ValueError("La vecindad quedó vacía después de filtrar por celdas de la malla.")

# Mapear a índices internos
w_use["i"] = w_use["cell_id"].map(cell_map)
w_use["j"] = w_use["neighbor_id"].map(cell_map)

if w_use["i"].isna().any() or w_use["j"].isna().any():
    raise ValueError("Hay relaciones de vecindad que no pudieron mapearse a índices internos.")

w_use["i"] = w_use["i"].astype(int)
w_use["j"] = w_use["j"].astype(int)

# Quitar lazos propios si existieran
w_use = w_use.loc[w_use["i"] != w_use["j"]].copy()

# Dejar pares únicos no dirigidos
ii = np.minimum(w_use["i"].values, w_use["j"].values)
jj = np.maximum(w_use["i"].values, w_use["j"].values)

pairs = np.unique(np.column_stack([ii, jj]), axis=0)
w_edges = pd.DataFrame(pairs, columns=["i", "j"])

# Guardar W final
w_edges_path = OUT_DIR / "HBM_NO2_W_edges_queen_v2.csv"
w_edges.to_csv(w_edges_path, index=False)

# Guardar folds
folds_path = OUT_DIR / "HBM_NO2_folds_v2.json"
with open(folds_path, "w", encoding="utf-8") as f:
    json.dump(FOLDS, f, ensure_ascii=False, indent=2)

# Resumen
print("Resumen CELDA 3")
print("- relaciones originales en W      :", len(w))
print("- relaciones válidas tras filtro  :", len(w_use))
print("- edges únicos no dirigidos       :", len(w_edges))
print("- nodos únicos en W               :", len(set(w_use['i']).union(set(w_use['j']))))

print("\nArchivos guardados:")
print("-", w_edges_path)
print("-", folds_path)

print("\nFolds temporales:")
print(json.dumps(FOLDS, ensure_ascii=False, indent=2))

Resumen CELDA 3
- relaciones originales en W      : 1610
- relaciones válidas tras filtro  : 1610
- edges únicos no dirigidos       : 805
- nodos únicos en W               : 254

Archivos guardados:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_W_edges_queen_v2.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_folds_v2.json

Folds temporales:
{
  "fold_1": {
    "train_years": [
      2020,
      2021
    ],
    "test_years": [
      2022
    ]
  },
  "fold_2": {
    "train_years": [
      2020,
      2021,
      2022
    ],
    "test_years": [
      2023
    ]
  },
  "fold_3": {
    "train_years": [
      2020,
      2021,
      2022,
      2023
    ],
    "test_years": [
      2024
    ]
  }
}



# # CELDA 4 — Funciones auxiliares para escalamiento, métricas e intervalos

# **Objetivo:**
definir funciones reutilizables para preparar los datos del HBM v2
y evaluar sus predicciones de manera consistente.

**Qué hace esta celda:**
 1. Define una función para estandarizar covariables usando solo los años de entrenamiento.
 2. Define una función para llevar esas covariables escaladas al panel observado.
 3. Define una función para calcular métricas puntuales:
    - MAE
    - RMSE
    - sesgo
    - correlación
    - R²
 4. Define métricas probabilísticas:
    - cobertura del intervalo 90%
    - ancho promedio del intervalo
    - WIS
 5. Define una función para extraer `p05`, `p50` y `p95`
    a partir del posterior del modelo.

 **Importancia metodológica:**
 el escalamiento se hace usando únicamente el período de entrenamiento,
 para evitar fuga de información hacia los folds de prueba.

In [5]:
# %%
# =========================
# CELDA 4) Funciones auxiliares
# =========================

def scale_grid_by_train_years(grid_df, x_cols, train_years):
    """
    Estandariza covariables usando solo las filas de grid_df
    pertenecientes a los años de entrenamiento.

    Parámetros
    ----------
    grid_df : pd.DataFrame
        Panel celda-mes de la malla.
    x_cols : list[str]
        Lista de covariables crudas a escalar.
    train_years : list[int]
        Años que pertenecen al conjunto de entrenamiento.

    Retorna
    -------
    grid_scaled : pd.DataFrame
        DataFrame con nuevas columnas *_z.
    params : dict
        Media y desviación estándar usadas para cada covariable.
    """
    grid_scaled = grid_df.copy()
    params = {}

    train_mask = grid_scaled["year"].isin(train_years)

    for col in x_cols:
        mu = grid_scaled.loc[train_mask, col].mean()
        sd = grid_scaled.loc[train_mask, col].std(ddof=0)

        if pd.isna(sd) or sd == 0:
            sd = 1.0

        z_col = f"{col}_z"
        grid_scaled[z_col] = (grid_scaled[col] - mu) / sd
        params[col] = {"mu": float(mu), "sd": float(sd)}

    return grid_scaled, params


def merge_scaled_covariates_to_obs(obs_df, grid_scaled, x_cols):
    """
    Lleva las covariables estandarizadas desde la malla al panel observado,
    usando la llave (cell_id, fecha).
    """
    z_cols = [f"{c}_z" for c in x_cols]

    out = obs_df.drop(columns=x_cols, errors="ignore").merge(
        grid_scaled[["cell_id", "fecha"] + z_cols],
        on=["cell_id", "fecha"],
        how="left",
        validate="many_to_one"
    )

    missing = out[z_cols].isna().any(axis=1).sum()
    if missing > 0:
        raise ValueError(f"Hay {missing} observaciones sin covariables escaladas.")

    return out


def interval_metrics(y_true, p05, p50, p95, alpha=0.10):
    """
    Calcula métricas puntuales y probabilísticas
    sobre un intervalo central del 90%.
    """
    y_true = np.asarray(y_true, dtype=float)
    p05 = np.asarray(p05, dtype=float)
    p50 = np.asarray(p50, dtype=float)
    p95 = np.asarray(p95, dtype=float)

    m = ~np.isnan(y_true) & ~np.isnan(p05) & ~np.isnan(p50) & ~np.isnan(p95)
    y_true = y_true[m]
    p05 = p05[m]
    p50 = p50[m]
    p95 = p95[m]

    if len(y_true) == 0:
        return {
            "n": 0,
            "mae": np.nan,
            "rmse": np.nan,
            "bias": np.nan,
            "r": np.nan,
            "r2": np.nan,
            "coverage_90": np.nan,
            "width_90_mean": np.nan,
            "wis_90": np.nan,
        }

    err = p50 - y_true
    mae = np.mean(np.abs(err))
    rmse = np.sqrt(np.mean(err**2))
    bias = np.mean(err)

    if np.std(y_true) == 0 or np.std(p50) == 0:
        r = np.nan
        r2 = np.nan
    else:
        r = np.corrcoef(y_true, p50)[0, 1]
        r2 = r**2

    coverage = np.mean((y_true >= p05) & (y_true <= p95))
    width = np.mean(p95 - p05)

    wis = np.mean(
        (p95 - p05)
        + (2 / alpha) * (p05 - y_true) * (y_true < p05)
        + (2 / alpha) * (y_true - p95) * (y_true > p95)
    )

    return {
        "n": int(len(y_true)),
        "mae": float(mae),
        "rmse": float(rmse),
        "bias": float(bias),
        "r": float(r) if not np.isnan(r) else np.nan,
        "r2": float(r2) if not np.isnan(r2) else np.nan,
        "coverage_90": float(coverage),
        "width_90_mean": float(width),
        "wis_90": float(wis),
    }


def posterior_predict_concentration(idata, X_mat, cell_idx_arr, time_idx_arr, eps=1e-6):
    """
    A partir del posterior del HBM, obtiene p05, p50 y p95
    en escala original de concentración.
    """
    alpha_s = idata.posterior["alpha"].values.reshape(-1)
    beta_s = idata.posterior["beta"].values.reshape(-1, idata.posterior["beta"].values.shape[-1])
    phi_s = idata.posterior["phi"].values.reshape(-1, idata.posterior["phi"].values.shape[-1])
    delta_s = idata.posterior["delta"].values.reshape(-1, idata.posterior["delta"].values.shape[-1])

    mu_s = (
        alpha_s[:, None]
        + (beta_s @ X_mat.T)
        + phi_s[:, cell_idx_arr]
        + delta_s[:, time_idx_arr]
    )

    c_s = np.exp(mu_s) - eps

    p05 = np.quantile(c_s, 0.05, axis=0)
    p50 = np.quantile(c_s, 0.50, axis=0)
    p95 = np.quantile(c_s, 0.95, axis=0)

    return p05, p50, p95


print("CELDA 4 cargada correctamente.")
print("Funciones disponibles:")
print("- scale_grid_by_train_years")
print("- merge_scaled_covariates_to_obs")
print("- interval_metrics")
print("- posterior_predict_concentration")

CELDA 4 cargada correctamente.
Funciones disponibles:
- scale_grid_by_train_years
- merge_scaled_covariates_to_obs
- interval_metrics
- posterior_predict_concentration



 # CELDA 5 — Definición del modelo base M1: ICAR + RW1

 **Objetivo:**
 dejar definida la función que ajusta el HBM base para un fold temporal.

 **Especificación del modelo M1:**

 - Respuesta observada:
   `NO2_obs`
 - Escala:
   logarítmica
 - Efectos fijos:
   covariables A–D + viento
 - Efecto espacial:
   `ICAR`
 - Efecto temporal:
   `RW1`

 **Forma general del modelo:**

 `log(NO2_obs) = alpha + X beta + phi_i + delta_t + error`

 donde:
 - `phi_i` representa el efecto espacial estructurado sobre la malla
 - `delta_t` representa la evolución temporal mensual

 **Qué hace esta celda:**
 1. Importa PyMC, PyTensor y ArviZ.
 2. Define la función `fit_hbm_m1_fold`.
 3. La función:
    - recibe train/test por años,
    - ajusta el modelo bayesiano,
    - predice sobre el período test,
    - calcula percentiles posteriores,
    - devuelve métricas e intervalos.

 **Nota:**
 si PyMC no está instalado en tu entorno, esta celda te lo indicará.

In [6]:
# %%
# =========================
# CELDA 5) Modelo base M1: ICAR + RW1
# =========================

try:
    import pymc as no
    import pytensor.tensor as pt
    import arviz as az
except Exception as e:
    raise ImportError(
        "No se pudieron importar pymc / pytensor / arviz.\n"
        "Instala en tu entorno:\n"
        "pip install pymc arviz pytensor\n\n"
        f"Detalle original: {e}"
    )


def fit_hbm_m1_fold(
    obs_scaled,
    grid_scaled,
    w_edges_df,
    x_cols_raw,
    train_years,
    test_years,
    n_cells,
    n_times,
    draws=1000,
    tune=1000,
    chains=4,
    target_accept=0.95,
    random_seed=42,
):
    """
    Ajusta el HBM base M1 para un fold temporal:
    - espacial ICAR
    - temporal RW1
    - respuesta observada NO2_obs
    """

    z_cols = [f"{c}_z" for c in x_cols_raw]

    train_mask_obs = obs_scaled["year"].isin(train_years)
    test_mask_obs = obs_scaled["year"].isin(test_years)

    train_obs = obs_scaled.loc[train_mask_obs].copy()
    test_obs = obs_scaled.loc[test_mask_obs].copy()

    if len(train_obs) == 0:
        raise ValueError("No hay observaciones de entrenamiento para este fold.")
    if len(test_obs) == 0:
        raise ValueError("No hay observaciones de prueba para este fold.")

    # Matrices y vectores del conjunto de entrenamiento
    X_train = train_obs[z_cols].to_numpy(dtype=float)
    y_train_raw = train_obs["NO2_obs"].to_numpy(dtype=float)
    y_train = np.log(y_train_raw + EPS) if USE_LOG else y_train_raw

    cell_train = train_obs["cell_idx"].to_numpy(dtype=int)
    time_train = train_obs["time_idx"].to_numpy(dtype=int)

    # Matrices y vectores del conjunto de prueba
    X_test = test_obs[z_cols].to_numpy(dtype=float)
    y_test = test_obs["NO2_obs"].to_numpy(dtype=float)
    cell_test = test_obs["cell_idx"].to_numpy(dtype=int)
    time_test = test_obs["time_idx"].to_numpy(dtype=int)

    # Edges espaciales
    ei = w_edges_df["i"].to_numpy(dtype=int)
    ej = w_edges_df["j"].to_numpy(dtype=int)

    p = X_train.shape[1]

    with no.Model() as model:
        # -------------------------
        # Efectos fijos
        # -------------------------
        alpha = no.Normal("alpha", mu=0.0, sigma=5.0)
        beta = no.Normal("beta", mu=0.0, sigma=1.0, shape=p)

        # -------------------------
        # Error observacional
        # -------------------------
        sigma_y = no.HalfNormal("sigma_y", sigma=1.0)

        # -------------------------
        # Efecto espacial ICAR
        # -------------------------
        tau_phi = no.Exponential("tau_phi", 1.0)
        phi_raw = no.Normal("phi_raw", mu=0.0, sigma=1.0, shape=n_cells)
        phi = no.Deterministic("phi", phi_raw - pt.mean(phi_raw))

        no.Potential(
            "icar_penalty",
            -0.5 * tau_phi * pt.sum((phi[ei] - phi[ej]) ** 2)
        )

        # -------------------------
        # Efecto temporal RW1
        # -------------------------
        sigma_t = no.HalfNormal("sigma_t", sigma=1.0)
        delta_raw = no.GaussianRandomWalk("delta_raw", sigma=sigma_t, shape=n_times)
        delta = no.Deterministic("delta", delta_raw - pt.mean(delta_raw))

        # -------------------------
        # Media del modelo
        # -------------------------
        mu_train = alpha + pt.dot(X_train, beta) + phi[cell_train] + delta[time_train]

        # -------------------------
        # Verosimilitud
        # -------------------------
        no.Normal("y_obs", mu=mu_train, sigma=sigma_y, observed=y_train)

        # -------------------------
        # Muestreo MCMC
        # -------------------------
        idata = no.sample(
            draws=draws,
            tune=tune,
            chains=chains,
            target_accept=target_accept,
            random_seed=random_seed,
            return_inferencedata=True,
            progressbar=True,
        )

    # -------------------------
    # Predicción sobre test
    # -------------------------
    p05_test, p50_test, p95_test = posterior_predict_concentration(
        idata=idata,
        X_mat=X_test,
        cell_idx_arr=cell_test,
        time_idx_arr=time_test,
        eps=EPS
    )

    test_pred = test_obs[
        ["Estacion", "fecha", "year", "month", "cell_id", "cell_idx", "time_idx", "NO2_obs"]
    ].copy()

    test_pred["p05_hbm"] = p05_test
    test_pred["p50_hbm"] = p50_test
    test_pred["p95_hbm"] = p95_test
    test_pred["width_90"] = test_pred["p95_hbm"] - test_pred["p05_hbm"]

    # -------------------------
    # Métricas del fold
    # -------------------------
    met = interval_metrics(
        y_true=test_pred["NO2_obs"].values,
        p05=test_pred["p05_hbm"].values,
        p50=test_pred["p50_hbm"].values,
        p95=test_pred["p95_hbm"].values,
        alpha=0.10
    )

    return idata, test_pred, met


print("CELDA 5 cargada correctamente.")
print("Función disponible: fit_hbm_m1_fold")

CELDA 5 cargada correctamente.
Función disponible: fit_hbm_m1_fold



 # CELDA 6 — Validación temporal del modelo base M1

 **Objetivo:**
 ejecutar la validación temporal del HBM base usando los 3 folds definidos.

 **Qué hace esta celda:**
 1. Recorre cada fold temporal.
 2. Escala las covariables usando únicamente los años de entrenamiento.
 3. Lleva las covariables escaladas al panel observado.
 4. Ajusta el modelo HBM base M1:
    - espacial ICAR
    - temporal RW1
 5. Predice sobre el período de prueba.
 6. Guarda:
    - predicciones del fold
    - resumen del posterior
    - parámetros de escalamiento
 7. Consolida:
    - métricas de todos los folds
    - predicciones conjuntas de prueba

 **Salidas principales:**
 - `HBM_NO2_M1_fold_1_pred_test.csv`
 - `HBM_NO2_M1_fold_2_pred_test.csv`
 - `HBM_NO2_M1_fold_3_pred_test.csv`
 - `HBM_NO2_M1_metrics_folds.csv`
 - `HBM_NO2_M1_pred_test_all_folds.csv`

 **Nota práctica:**
 esta es la primera corrida real del HBM. Puede tardar bastante
 dependiendo del equipo y del entorno de Python.

In [7]:
# %%
# =========================
# CELDA 6) Validación temporal M1
# =========================

all_metrics = []
all_test_preds = []

# puedes subir estos valores más adelante si quieres una corrida más exigente
DRAWS = 800
TUNE = 800
CHAINS = 4
TARGET_ACCEPT = 0.95
RANDOM_SEED = 42

for fold_name, fold_info in FOLDS.items():
    print("\n" + "=" * 80)
    print(f"Corriendo {fold_name}")
    print("Train years:", fold_info["train_years"])
    print("Test years :", fold_info["test_years"])

    # -------------------------------------------------
    # 1) Escalamiento usando solo años train
    # -------------------------------------------------
    grid_scaled, scale_params = scale_grid_by_train_years(
        grid_df=grid_base,
        x_cols=X_COLS_RAW,
        train_years=fold_info["train_years"]
    )

    # -------------------------------------------------
    # 2) Llevar covariables escaladas al panel observado
    # -------------------------------------------------
    obs_scaled = merge_scaled_covariates_to_obs(
        obs_df=obs_panel,
        grid_scaled=grid_scaled,
        x_cols=X_COLS_RAW
    )

    # -------------------------------------------------
    # 3) Ajustar modelo del fold
    # -------------------------------------------------
    idata_fold, test_pred_fold, met_fold = fit_hbm_m1_fold(
        obs_scaled=obs_scaled,
        grid_scaled=grid_scaled,
        w_edges_df=w_edges,
        x_cols_raw=X_COLS_RAW,
        train_years=fold_info["train_years"],
        test_years=fold_info["test_years"],
        n_cells=len(cell_ids),
        n_times=len(time_ids),
        draws=DRAWS,
        tune=TUNE,
        chains=CHAINS,
        target_accept=TARGET_ACCEPT,
        random_seed=RANDOM_SEED,
    )

    # -------------------------------------------------
    # 4) Guardar predicciones del fold
    # -------------------------------------------------
    test_pred_fold["fold"] = fold_name
    pred_fold_path = OUT_DIR / f"HBM_NO2_M1_{fold_name}_pred_test.csv"
    test_pred_fold.to_csv(pred_fold_path, index=False)

    # -------------------------------------------------
    # 5) Guardar resumen del posterior
    # -------------------------------------------------
    summary_fold = az.summary(
        idata_fold,
        var_names=["alpha", "beta", "sigma_y", "tau_phi", "sigma_t"],
        round_to=4
    )
    summary_fold_path = OUT_DIR / f"HBM_NO2_M1_{fold_name}_summary.csv"
    summary_fold.to_csv(summary_fold_path)

    # -------------------------------------------------
    # 6) Guardar parámetros de escalamiento
    # -------------------------------------------------
    scale_fold_path = OUT_DIR / f"HBM_NO2_M1_{fold_name}_scale_params.json"
    with open(scale_fold_path, "w", encoding="utf-8") as f:
        json.dump(scale_params, f, ensure_ascii=False, indent=2)

    # -------------------------------------------------
    # 7) Consolidar métricas
    # -------------------------------------------------
    met_fold["fold"] = fold_name
    met_fold["train_years"] = ",".join(map(str, fold_info["train_years"]))
    met_fold["test_years"] = ",".join(map(str, fold_info["test_years"]))
    all_metrics.append(met_fold)
    all_test_preds.append(test_pred_fold)

    print("\nGuardado del fold:")
    print("-", pred_fold_path)
    print("-", summary_fold_path)
    print("-", scale_fold_path)

    print("\nMétricas del fold:")
    for k, v in met_fold.items():
        print(f"- {k}: {v}")

# -------------------------------------------------
# 8) Consolidar todo
# -------------------------------------------------
metrics_df = pd.DataFrame(all_metrics)
preds_df = pd.concat(all_test_preds, ignore_index=True)

metrics_path = OUT_DIR / "HBM_NO2_M1_metrics_folds.csv"
preds_path = OUT_DIR / "HBM_NO2_M1_pred_test_all_folds.csv"

metrics_df.to_csv(metrics_path, index=False)
preds_df.to_csv(preds_path, index=False)

print("\n" + "=" * 80)
print("VALIDACIÓN TEMPORAL COMPLETADA")
print("- métricas consolidadas :", metrics_path)
print("- predicciones consolidadas :", preds_path)

print("\nResumen final de métricas:")
display(metrics_df)


Corriendo fold_1
Train years: [2020, 2021]
Test years : [2022]


Initializing NUTS using jitter+adapt_diag...
ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: local_subtensor_merge
ERROR (pytensor.graph.rewriting.basic): node: Subtensor{i}(Subtensor{start:}.0, 0)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 1920, in process_node
    replacements = node_rewriter.transform(
                   ^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 993, in transform
    return self.fn(fgraph, node)
           ^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 416, in local_subtensor_merge
    merge_two_slices(
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 

Output()

Sampling 4 chains for 800 tune and 800 draw iterations (3_200 + 3_200 draws total) took 143 seconds.
Chain 1 reached the maximum tree depth. Increase `max_treedepth`, increase `target_accept` or reparameterize.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



Guardado del fold:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1_fold_1_pred_test.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1_fold_1_summary.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1_fold_1_scale_params.json

Métricas del fold:
- n: 155
- mae: 2.507006824183464
- rmse: 3.0942494406826118
- bias: -0.40029528364939543
- r: 0.7758845347833678
- r2: 0.6019968113160031
- coverage_90: 0.9806451612903225
- width_90_mean: 22.484900404917674
- wis_90: 23.039218621620236
- fold: fold_1
- train_years: 2020,2021
- test_years: 2022

Corriendo fold_2
Train years: [2020, 2021, 2022]
Test years : [2023]


...

Initializing NUTS using jitter+adapt_diag...
ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: local_subtensor_merge
ERROR (pytensor.graph.rewriting.basic): node: Subtensor{i}(Subtensor{start:}.0, 0)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 1920, in process_node
    replacements = node_rewriter.transform(
                   ^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 993, in transform
    return self.fn(fgraph, node)
           ^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 416, in local_subtensor_merge
    merge_two_slices(
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 

Output()

Sampling 4 chains for 800 tune and 800 draw iterations (3_200 + 3_200 draws total) took 238 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



Guardado del fold:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1_fold_3_pred_test.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1_fold_3_summary.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1_fold_3_scale_params.json

Métricas del fold:
- n: 144
- mae: 2.3153636416774663
- rmse: 2.8262491219923924
- bias: 0.7515177840796241
- r: 0.8543652619035861
- r2: 0.7299400007475833
- coverage_90: 0.9722222222222222
- width_90_mean: 19.20016891488419
- wis_90: 19.596887747001592
- fold: fold_3
- train_years: 2020,2021,2022,2023
- test_years: 2024

VALIDACIÓN TEMPORAL COMPLETADA
- métricas consolidadas : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1_metrics_folds.csv
- predicciones consolidadas : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1_pred_test_all_folds.csv

Resumen final de métricas:


,n,mae,rmse,bias,r,r2,coverage_90,width_90_mean,wis_90,fold,train_years,test_years
0,155,2.507007,3.094249,-0.400295,0.775885,0.601997,0.980645,22.484900,23.039219,fold_1,"2020,2021",2022
1,154,2.313809,2.798398,0.856929,0.863356,0.745383,0.987013,20.805201,20.864105,fold_2,"2020,2021,2022",2023
2,144,2.315364,2.826249,0.751518,0.854365,0.729940,0.972222,19.200169,19.596888,fold_3,"2020,2021,2022,2023",2024



 # CELDA 6B — Redefinir el modelo base en versión más estable (M1b)

 **Objetivo:**
 crear una versión más robusta del HBM base para mejorar la convergencia
 sin cambiar la estructura sustantiva del modelo.

 **Qué cambia respecto al M1 inicial:**
 1. Se mantienen:
    - respuesta observada `NO2_obs`
    - efecto espacial ICAR
    - efecto temporal RW1
    - covariables A–D + viento
 2. Se ajustan priors para hacer el modelo más regularizado.
 3. Se mejora el muestreo con:
    - `target_accept` más alto
    - `max_treedepth` más alto
    - `init="adapt_diag"`

 **Objetivo metodológico:**
 reducir problemas de:
 - Rhat alto
 - ESS bajo
 - tree depth máximo

 **Importante:**
 esta celda solo redefine la función.
 En la siguiente la volvemos a correr por folds.

In [8]:
# %%
# =========================
# CELDA 6B) Versión estable del modelo base
# =========================

def fit_hbm_m1b_fold(
    obs_scaled,
    grid_scaled,
    w_edges_df,
    x_cols_raw,
    train_years,
    test_years,
    n_cells,
    n_times,
    draws=1000,
    tune=1500,
    chains=4,
    target_accept=0.99,
    max_treedepth=15,
    random_seed=42,
):
    """
    Ajusta el HBM base M1b para un fold temporal:
    - espacial ICAR
    - temporal RW1
    - respuesta observada NO2_obs
    - priors más regularizantes
    - sampler más conservador
    """

    z_cols = [f"{c}_z" for c in x_cols_raw]

    train_mask_obs = obs_scaled["year"].isin(train_years)
    test_mask_obs = obs_scaled["year"].isin(test_years)

    train_obs = obs_scaled.loc[train_mask_obs].copy()
    test_obs = obs_scaled.loc[test_mask_obs].copy()

    if len(train_obs) == 0:
        raise ValueError("No hay observaciones de entrenamiento para este fold.")
    if len(test_obs) == 0:
        raise ValueError("No hay observaciones de prueba para este fold.")

    # -------------------------
    # Datos train
    # -------------------------
    X_train = train_obs[z_cols].to_numpy(dtype=float)
    y_train_raw = train_obs["NO2_obs"].to_numpy(dtype=float)
    y_train = np.log(y_train_raw + EPS) if USE_LOG else y_train_raw

    cell_train = train_obs["cell_idx"].to_numpy(dtype=int)
    time_train = train_obs["time_idx"].to_numpy(dtype=int)

    # -------------------------
    # Datos test
    # -------------------------
    X_test = test_obs[z_cols].to_numpy(dtype=float)
    y_test = test_obs["NO2_obs"].to_numpy(dtype=float)
    cell_test = test_obs["cell_idx"].to_numpy(dtype=int)
    time_test = test_obs["time_idx"].to_numpy(dtype=int)

    # -------------------------
    # Vecindad espacial
    # -------------------------
    ei = w_edges_df["i"].to_numpy(dtype=int)
    ej = w_edges_df["j"].to_numpy(dtype=int)

    p = X_train.shape[1]
    alpha_mu = float(np.mean(y_train))

    with no.Model() as model:
        # -------------------------
        # Efectos fijos más regularizados
        # -------------------------
        alpha = no.Normal("alpha", mu=alpha_mu, sigma=1.0)
        beta = no.Normal("beta", mu=0.0, sigma=0.5, shape=p)

        # -------------------------
        # Error observacional
        # -------------------------
        sigma_y = no.HalfNormal("sigma_y", sigma=0.75)

        # -------------------------
        # Efecto espacial ICAR
        # -------------------------
        tau_phi = no.Exponential("tau_phi", 2.0)
        phi_raw = no.Normal("phi_raw", mu=0.0, sigma=1.0, shape=n_cells)
        phi = no.Deterministic("phi", phi_raw - pt.mean(phi_raw))

        no.Potential(
            "icar_penalty",
            -0.5 * tau_phi * pt.sum((phi[ei] - phi[ej]) ** 2)
        )

        # -------------------------
        # Efecto temporal RW1 más controlado
        # -------------------------
        sigma_t = no.HalfNormal("sigma_t", sigma=0.25)
        delta_raw = no.GaussianRandomWalk("delta_raw", sigma=sigma_t, shape=n_times)
        delta = no.Deterministic("delta", delta_raw - pt.mean(delta_raw))

        # -------------------------
        # Media del modelo
        # -------------------------
        mu_train = alpha + pt.dot(X_train, beta) + phi[cell_train] + delta[time_train]

        # -------------------------
        # Likelihood
        # -------------------------
        no.Normal("y_obs", mu=mu_train, sigma=sigma_y, observed=y_train)

        # -------------------------
        # Muestreo
        # -------------------------
        idata = no.sample(
            draws=draws,
            tune=tune,
            chains=chains,
            init="adapt_diag",
            target_accept=target_accept,
            max_treedepth=max_treedepth,
            random_seed=random_seed,
            return_inferencedata=True,
            progressbar=True,
        )

    # -------------------------
    # Predicción test en escala original
    # -------------------------
    p05_test, p50_test, p95_test = posterior_predict_concentration(
        idata=idata,
        X_mat=X_test,
        cell_idx_arr=cell_test,
        time_idx_arr=time_test,
        eps=EPS
    )

    test_pred = test_obs[
        ["Estacion", "fecha", "year", "month", "cell_id", "cell_idx", "time_idx", "NO2_obs"]
    ].copy()

    test_pred["p05_hbm"] = p05_test
    test_pred["p50_hbm"] = p50_test
    test_pred["p95_hbm"] = p95_test
    test_pred["width_90"] = test_pred["p95_hbm"] - test_pred["p05_hbm"]

    met = interval_metrics(
        y_true=test_pred["NO2_obs"].values,
        p05=test_pred["p05_hbm"].values,
        p50=test_pred["p50_hbm"].values,
        p95=test_pred["p95_hbm"].values,
        alpha=0.10
    )

    return idata, test_pred, met


print("CELDA 6B cargada correctamente.")
print("Función disponible: fit_hbm_m1b_fold")

CELDA 6B cargada correctamente.
Función disponible: fit_hbm_m1b_fold


# CELDA 6C — Validación temporal del modelo estable M1b

 **Objetivo:**
 volver a ejecutar la validación temporal del HBM usando la versión
 más estable del modelo base:

 - espacial ICAR
 - temporal RW1
 - priors más regularizantes
 - muestreo más conservador

 **Qué hace esta celda:**
 1. Recorre los 3 folds temporales.
 2. Escala covariables usando solo train.
 3. Ajusta `fit_hbm_m1b_fold`.
 4. Guarda:
    - predicciones por fold
   - resumen posterior por fold
    - parámetros de escalamiento
 5. Consolida:
    - métricas de todos los folds
    - predicciones de prueba conjuntas
 6. Si existe el archivo de métricas del modelo M1 anterior,
    genera una tabla comparativa M1 vs M1b.

 **Salidas principales:**
 - `HBM_NO2_M1b_fold_1_pred_test.csv`
 - `HBM_NO2_M1b_fold_2_pred_test.csv`
 - `HBM_NO2_M1b_fold_3_pred_test.csv`
 - `HBM_NO2_M1b_metrics_folds.csv`
 - `HBM_NO2_M1b_pred_test_all_folds.csv`
 - `HBM_NO2_compare_M1_vs_M1b.csv` (si existe M1)

 **Qué esperamos observar:**
 - menor problema de convergencia
 - Rhat más cercano a 1
 - ESS más alto
 - calibración de intervalos más estable

In [9]:
# %%
# =========================
# CELDA 6C) Validación temporal M1b
# =========================

all_metrics_m1b = []
all_test_preds_m1b = []

# configuración más conservadora
DRAWS_B = 1000
TUNE_B = 1500
CHAINS_B = 4
TARGET_ACCEPT_B = 0.99
MAX_TREEDEPTH_B = 15
RANDOM_SEED_B = 42

for fold_name, fold_info in FOLDS.items():
    print("\n" + "=" * 80)
    print(f"Corriendo M1b - {fold_name}")
    print("Train years:", fold_info["train_years"])
    print("Test years :", fold_info["test_years"])

    # -------------------------------------------------
    # 1) Escalamiento usando solo train
    # -------------------------------------------------
    grid_scaled_b, scale_params_b = scale_grid_by_train_years(
        grid_df=grid_base,
        x_cols=X_COLS_RAW,
        train_years=fold_info["train_years"]
    )

    # -------------------------------------------------
    # 2) Llevar covariables escaladas al panel observado
    # -------------------------------------------------
    obs_scaled_b = merge_scaled_covariates_to_obs(
        obs_df=obs_panel,
        grid_scaled=grid_scaled_b,
        x_cols=X_COLS_RAW
    )

    # -------------------------------------------------
    # 3) Ajustar modelo del fold
    # -------------------------------------------------
    idata_fold_b, test_pred_fold_b, met_fold_b = fit_hbm_m1b_fold(
        obs_scaled=obs_scaled_b,
        grid_scaled=grid_scaled_b,
        w_edges_df=w_edges,
        x_cols_raw=X_COLS_RAW,
        train_years=fold_info["train_years"],
        test_years=fold_info["test_years"],
        n_cells=len(cell_ids),
        n_times=len(time_ids),
        draws=DRAWS_B,
        tune=TUNE_B,
        chains=CHAINS_B,
        target_accept=TARGET_ACCEPT_B,
        max_treedepth=MAX_TREEDEPTH_B,
        random_seed=RANDOM_SEED_B,
    )

    # -------------------------------------------------
    # 4) Guardar predicciones del fold
    # -------------------------------------------------
    test_pred_fold_b["fold"] = fold_name
    pred_fold_path_b = OUT_DIR / f"HBM_NO2_M1b_{fold_name}_pred_test.csv"
    test_pred_fold_b.to_csv(pred_fold_path_b, index=False)

    # -------------------------------------------------
    # 5) Guardar resumen del posterior
    # -------------------------------------------------
    summary_fold_b = az.summary(
        idata_fold_b,
        var_names=["alpha", "beta", "sigma_y", "tau_phi", "sigma_t"],
        round_to=4
    )
    summary_fold_path_b = OUT_DIR / f"HBM_NO2_M1b_{fold_name}_summary.csv"
    summary_fold_b.to_csv(summary_fold_path_b)

    # -------------------------------------------------
    # 6) Guardar parámetros de escalamiento
    # -------------------------------------------------
    scale_fold_path_b = OUT_DIR / f"HBM_NO2_M1b_{fold_name}_scale_params.json"
    with open(scale_fold_path_b, "w", encoding="utf-8") as f:
        json.dump(scale_params_b, f, ensure_ascii=False, indent=2)

    # -------------------------------------------------
    # 7) Consolidar métricas
    # -------------------------------------------------
    met_fold_b["model"] = "M1b"
    met_fold_b["fold"] = fold_name
    met_fold_b["train_years"] = ",".join(map(str, fold_info["train_years"]))
    met_fold_b["test_years"] = ",".join(map(str, fold_info["test_years"]))

    all_metrics_m1b.append(met_fold_b)
    all_test_preds_m1b.append(test_pred_fold_b)

    print("\nGuardado del fold:")
    print("-", pred_fold_path_b)
    print("-", summary_fold_path_b)
    print("-", scale_fold_path_b)

    print("\nMétricas del fold:")
    for k, v in met_fold_b.items():
        print(f"- {k}: {v}")

# -------------------------------------------------
# 8) Consolidar resultados M1b
# -------------------------------------------------
metrics_m1b_df = pd.DataFrame(all_metrics_m1b)
preds_m1b_df = pd.concat(all_test_preds_m1b, ignore_index=True)

metrics_m1b_path = OUT_DIR / "HBM_NO2_M1b_metrics_folds.csv"
preds_m1b_path = OUT_DIR / "HBM_NO2_M1b_pred_test_all_folds.csv"

metrics_m1b_df.to_csv(metrics_m1b_path, index=False)
preds_m1b_df.to_csv(preds_m1b_path, index=False)

print("\n" + "=" * 80)
print("VALIDACIÓN TEMPORAL M1b COMPLETADA")
print("- métricas consolidadas :", metrics_m1b_path)
print("- predicciones consolidadas :", preds_m1b_path)

print("\nResumen final de métricas M1b:")
display(metrics_m1b_df)

# -------------------------------------------------
# 9) Comparar M1 vs M1b si existe archivo anterior
# -------------------------------------------------
m1_metrics_path = OUT_DIR / "HBM_NO2_M1_metrics_folds.csv"

if m1_metrics_path.exists():
    metrics_m1_df = pd.read_csv(m1_metrics_path).copy()
    metrics_m1_df["model"] = "M1"

    cols_keep = [
        "model", "fold", "n", "mae", "rmse", "bias", "r", "r2",
        "coverage_90", "width_90_mean", "wis_90", "train_years", "test_years"
    ]

    compare_df = pd.concat(
        [
            metrics_m1_df[cols_keep],
            metrics_m1b_df[cols_keep]
        ],
        ignore_index=True
    )

    compare_path = OUT_DIR / "HBM_NO2_compare_M1_vs_M1b.csv"
    compare_df.to_csv(compare_path, index=False)

    print("\nComparación M1 vs M1b guardada en:")
    print("-", compare_path)

    print("\nTabla comparativa:")
    display(compare_df.sort_values(["fold", "model"]).reset_index(drop=True))
else:
    print("\nNo se encontró el archivo de métricas de M1 previo.")
    print("Se omitió la comparación M1 vs M1b.")


Corriendo M1b - fold_1
Train years: [2020, 2021]
Test years : [2022]


Initializing NUTS using adapt_diag...
ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: local_subtensor_merge
ERROR (pytensor.graph.rewriting.basic): node: Subtensor{i}(Subtensor{start:}.0, 0)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 1920, in process_node
    replacements = node_rewriter.transform(
                   ^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 993, in transform
    return self.fn(fgraph, node)
           ^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 416, in local_subtensor_merge
    merge_two_slices(
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 865, in

Output()

Sampling 4 chains for 1_500 tune and 1_000 draw iterations (6_000 + 4_000 draws total) took 2648 seconds.
There was 1 divergence after tuning. Increase `target_accept` or reparameterize.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



Guardado del fold:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1b_fold_1_pred_test.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1b_fold_1_summary.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1b_fold_1_scale_params.json

Métricas del fold:
- n: 155
- mae: 2.5062987820461897
- rmse: 3.083537858250615
- bias: -0.20495349167975732
- r: 0.7751846045323156
- r2: 0.6009111711039226
- coverage_90: 0.9870967741935484
- width_90_mean: 21.723953534449713
- wis_90: 22.28671941891343
- model: M1b
- fold: fold_1
- train_years: 2020,2021
- test_years: 2022

Corriendo M1b - fold_2
Train years: [2020, 2021, 2022]
Test years : [2023]


...

Sampling 4 chains for 1_500 tune and 1_000 draw iterations (6_000 + 4_000 draws total) took 2988 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



Guardado del fold:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1b_fold_3_pred_test.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1b_fold_3_summary.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1b_fold_3_scale_params.json

Métricas del fold:
- n: 144
- mae: 2.3468423299956616
- rmse: 2.88006129099493
- bias: 0.8467389988442084
- r: 0.851660852107935
- r2: 0.7253262070132138
- coverage_90: 0.9722222222222222
- width_90_mean: 19.332074406778133
- wis_90: 19.7719367992546
- model: M1b
- fold: fold_3
- train_years: 2020,2021,2022,2023
- test_years: 2024

VALIDACIÓN TEMPORAL M1b COMPLETADA
- métricas consolidadas : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1b_metrics_folds.csv
- predicciones consolidadas : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1b_pred_test_all_folds.csv

Resumen final de métricas M1b:


,n,mae,rmse,bias,r,r2,coverage_90,width_90_mean,wis_90,model,fold,train_years,test_years
0,155,2.506299,3.083538,-0.204953,0.775185,0.600911,0.987097,21.723954,22.286719,M1b,fold_1,"2020,2021",2022
1,154,2.322275,2.810962,0.868654,0.862314,0.743585,0.974026,20.609334,20.753718,M1b,fold_2,"2020,2021,2022",2023
2,144,2.346842,2.880061,0.846739,0.851661,0.725326,0.972222,19.332074,19.771937,M1b,fold_3,"2020,2021,2022,2023",2024



Comparación M1 vs M1b guardada en:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_compare_M1_vs_M1b.csv

Tabla comparativa:


,model,fold,n,mae,rmse,bias,r,r2,coverage_90,width_90_mean,wis_90,train_years,test_years
0,M1,fold_1,155,2.507007,3.094249,-0.400295,0.775885,0.601997,0.980645,22.484900,23.039219,"2020,2021",2022
1,M1b,fold_1,155,2.506299,3.083538,-0.204953,0.775185,0.600911,0.987097,21.723954,22.286719,"2020,2021",2022
2,M1,fold_2,154,2.313809,2.798398,0.856929,0.863356,0.745383,0.987013,20.805201,20.864105,"2020,2021,2022",2023
3,M1b,fold_2,154,2.322275,2.810962,0.868654,0.862314,0.743585,0.974026,20.609334,20.753718,"2020,2021,2022",2023
4,M1,fold_3,144,2.315364,2.826249,0.751518,0.854365,0.729940,0.972222,19.200169,19.596888,"2020,2021,2022,2023",2024
5,M1b,fold_3,144,2.346842,2.880061,0.846739,0.851661,0.725326,0.972222,19.332074,19.771937,"2020,2021,2022,2023",2024



# CELDA 6D — Comparación de diagnósticos de convergencia: M1 vs M1b

 **Objetivo:**
 comparar formalmente la calidad del muestreo bayesiano entre los modelos
 M1 y M1b usando los archivos `summary.csv` guardados por fold.

 **Qué hace esta celda:**
 1. Lee los archivos resumen de:
    - `HBM_NO2_M1_fold_*_summary.csv`
    - `HBM_NO2_M1b_fold_*_summary.csv`
 2. Extrae, para cada fold:
    - máximo `r_hat`
    - mínimo `ess_bulk`
    - mínimo `ess_tail`
 3. Consolida la comparación M1 vs M1b.
 4. Marca reglas simples de interpretación:
    - `r_hat_ok`: max r_hat <= 1.01
    - `ess_bulk_ok`: min ess_bulk >= 400
    - `ess_tail_ok`: min ess_tail >= 400

 **Interpretación esperada:**
 - un buen modelo debe tener `r_hat` cercano a 1
 - ESS no debería ser muy bajo
 - si M1b mejora claramente estos indicadores, será preferible a M1
   aunque las métricas predictivas sean parecidas

In [10]:
# %%
# =========================
# CELDA 6D) Diagnósticos M1 vs M1b
# =========================

from pathlib import Path

summary_files = {
    "M1": {
        "fold_1": OUT_DIR / "HBM_NO2_M1_fold_1_summary.csv",
        "fold_2": OUT_DIR / "HBM_NO2_M1_fold_2_summary.csv",
        "fold_3": OUT_DIR / "HBM_NO2_M1_fold_3_summary.csv",
    },
    "M1b": {
        "fold_1": OUT_DIR / "HBM_NO2_M1b_fold_1_summary.csv",
        "fold_2": OUT_DIR / "HBM_NO2_M1b_fold_2_summary.csv",
        "fold_3": OUT_DIR / "HBM_NO2_M1b_fold_3_summary.csv",
    }
}

rows = []

for model_name, model_files in summary_files.items():
    for fold_name, path_summary in model_files.items():
        if not path_summary.exists():
            print(f"No existe: {path_summary}")
            continue

        df_sum = pd.read_csv(path_summary, index_col=0)

        # por seguridad, revisar que las columnas existan
        needed_cols = ["r_hat", "ess_bulk", "ess_tail"]
        for c in needed_cols:
            if c not in df_sum.columns:
                raise ValueError(f"Falta la columna '{c}' en {path_summary.name}")

        row = {
            "model": model_name,
            "fold": fold_name,
            "max_r_hat": df_sum["r_hat"].max(),
            "min_ess_bulk": df_sum["ess_bulk"].min(),
            "min_ess_tail": df_sum["ess_tail"].min(),
        }

        row["r_hat_ok"] = row["max_r_hat"] <= 1.01
        row["ess_bulk_ok"] = row["min_ess_bulk"] >= 400
        row["ess_tail_ok"] = row["min_ess_tail"] >= 400

        rows.append(row)

diag_compare = pd.DataFrame(rows).sort_values(["fold", "model"]).reset_index(drop=True)

diag_compare_path = OUT_DIR / "HBM_NO2_compare_diagnostics_M1_vs_M1b.csv"
diag_compare.to_csv(diag_compare_path, index=False)

print("Diagnósticos comparativos guardados en:")
print("-", diag_compare_path)

print("\nTabla comparativa de convergencia:")
display(diag_compare)

Diagnósticos comparativos guardados en:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_compare_diagnostics_M1_vs_M1b.csv

Tabla comparativa de convergencia:


,model,fold,max_r_hat,min_ess_bulk,min_ess_tail,r_hat_ok,ess_bulk_ok,ess_tail_ok
0,M1,fold_1,1.0047,344.4990,660.0106,True,False,True
1,M1b,fold_1,1.0034,501.6503,925.9753,True,True,True
2,M1,fold_2,1.0064,475.4902,965.8664,True,True,True
3,M1b,fold_2,1.0021,472.5702,1150.8206,True,True,True
4,M1,fold_3,1.0122,257.4289,392.0099,False,False,False
5,M1b,fold_3,1.0023,546.4780,943.2620,True,True,True



# CELDA 9A — Auditoría de calidad del panel observado no2.5

 **Objetivo:**
 revisar la calidad del panel observado de estaciones antes de hacer limpieza
 o volver a ajustar el HBM.

 **Qué hace esta celda:**
 1. Lee el archivo corregido de estaciones.
 2. Construye la fecha mensual.
 3. Filtra observaciones válidas de no2.5.
 4. Revisa duplicados por estación y mes.
 5. Resume la distribución de no2.5 por estación:
    - número de observaciones
    - media
    - desviación estándar
    - percentiles
    - mínimo y máximo
 6. Marca observaciones atípicas por estación usando una regla robusta IQR.
 7. Genera una tabla con las filas sospechosas.
 8. Guarda archivos de auditoría para usarlos en la siguiente etapa de limpieza.

 **Importante:**
 esta celda no elimina datos.
 Solo identifica dónde puede estar entrando ruido al modelo.

In [11]:
# %%
# =========================
# CELDA 9A) Auditoría de calidad del panel observado no2.5
# =========================

obs_audit = pd.read_csv(OBS_CSV)

# -------------------------------------------------
# 1) Fecha mensual
# -------------------------------------------------
obs_audit["fecha"] = pd.to_datetime(
    dict(year=obs_audit["Año"].astype(int), month=obs_audit["Mes"].astype(int), day=1),
    errors="coerce"
)

if obs_audit["fecha"].isna().any():
    raise ValueError("Hay fechas inválidas en el panel observado.")

# -------------------------------------------------
# 2) Filtrar no2.5 válido
# -------------------------------------------------
obs_audit["valido_NO2"] = obs_audit["valido_NO2"].astype(bool)

no_obs = obs_audit.loc[
    (obs_audit["valido_NO2"] == True) &
    (obs_audit["NO2"].notna())
].copy()

no_obs = no_obs.rename(columns={"NO2": "NO2_obs"})

print("Resumen general del panel observado no2.5")
print("- filas totales en archivo            :", len(obs_audit))
print("- filas válidas no2.5                :", len(no_obs))
print("- estaciones con no2.5 válido        :", no_obs["Estacion"].nunique())
print("- meses observados no2.5             :", no_obs["fecha"].nunique())
print("- rango temporal                     :", no_obs["fecha"].min().date(), "a", no_obs["fecha"].max().date())

# -------------------------------------------------
# 3) Duplicados por estación-mes
# -------------------------------------------------
dup_mask = no_obs.duplicated(subset=["Estacion", "fecha"], keep=False)
dup_rows = no_obs.loc[dup_mask].sort_values(["Estacion", "fecha"]).copy()

print("\nDuplicados por estación-mes:")
print("- número de filas duplicadas:", len(dup_rows))
print("- número de combinaciones duplicadas:",
      dup_rows[["Estacion", "fecha"]].drop_duplicates().shape[0])

# -------------------------------------------------
# 4) Resumen por estación
# -------------------------------------------------
station_summary = (
    no_obs.groupby("Estacion")["NO2_obs"]
    .agg(
        n="count",
        mean="mean",
        std="std",
        min="min",
        q01=lambda s: s.quantile(0.01),
        q05=lambda s: s.quantile(0.05),
        q25=lambda s: s.quantile(0.25),
        median="median",
        q75=lambda s: s.quantile(0.75),
        q95=lambda s: s.quantile(0.95),
        q99=lambda s: s.quantile(0.99),
        max="max",
    )
    .reset_index()
    .sort_values("mean", ascending=False)
)

# -------------------------------------------------
# 5) Regla robusta IQR por estación
# -------------------------------------------------
bounds = (
    no_obs.groupby("Estacion")["NO2_obs"]
    .agg(
        q1=lambda s: s.quantile(0.25),
        q3=lambda s: s.quantile(0.75)
    )
    .reset_index()
)

bounds["iqr"] = bounds["q3"] - bounds["q1"]
bounds["lower_iqr15"] = bounds["q1"] - 1.5 * bounds["iqr"]
bounds["upper_iqr15"] = bounds["q3"] + 1.5 * bounds["iqr"]
bounds["lower_iqr30"] = bounds["q1"] - 3.0 * bounds["iqr"]
bounds["upper_iqr30"] = bounds["q3"] + 3.0 * bounds["iqr"]

no_obs = no_obs.merge(bounds, on="Estacion", how="left")

no_obs["flag_iqr15"] = (
    (no_obs["NO2_obs"] < no_obs["lower_iqr15"]) |
    (no_obs["NO2_obs"] > no_obs["upper_iqr15"])
)

no_obs["flag_iqr30"] = (
    (no_obs["NO2_obs"] < no_obs["lower_iqr30"]) |
    (no_obs["NO2_obs"] > no_obs["upper_iqr30"])
)

# -------------------------------------------------
# 6) Tabla de observaciones sospechosas
# -------------------------------------------------
flagged_rows = no_obs.loc[
    no_obs["flag_iqr15"] == True,
    [
        "Estacion", "fecha", "Año", "Mes", "NO2_obs",
        "Temp_media", "HR", "Vel_viento", "Presión", "Precipitación",
        "q1", "q3", "iqr", "lower_iqr15", "upper_iqr15", "flag_iqr15", "flag_iqr30"
    ]
].sort_values(["Estacion", "fecha"])

# -------------------------------------------------
# 7) Resumen de flags por estación
# -------------------------------------------------
flag_summary = (
    no_obs.groupby("Estacion")
    .agg(
        n_total=("NO2_obs", "count"),
        n_flag_iqr15=("flag_iqr15", "sum"),
        n_flag_iqr30=("flag_iqr30", "sum"),
        NO2_mean=("NO2_obs", "mean"),
        NO2_std=("NO2_obs", "std"),
        NO2_min=("NO2_obs", "min"),
        NO2_max=("NO2_obs", "max")
    )
    .reset_index()
)

flag_summary["pct_flag_iqr15"] = 100 * flag_summary["n_flag_iqr15"] / flag_summary["n_total"]
flag_summary["pct_flag_iqr30"] = 100 * flag_summary["n_flag_iqr30"] / flag_summary["n_total"]

flag_summary = flag_summary.sort_values(["pct_flag_iqr15", "n_flag_iqr15"], ascending=False)

# -------------------------------------------------
# 8) Guardar auditoría
# -------------------------------------------------
station_summary_path = OUT_DIR / "HBM_NO2_obs_audit_station_summary.csv"
duplicates_path = OUT_DIR / "HBM_NO2_obs_audit_duplicates.csv"
flagged_rows_path = OUT_DIR / "HBM_NO2_obs_audit_flagged_rows.csv"
flag_summary_path = OUT_DIR / "HBM_NO2_obs_audit_flag_summary.csv"

station_summary.to_csv(station_summary_path, index=False)
dup_rows.to_csv(duplicates_path, index=False)
flagged_rows.to_csv(flagged_rows_path, index=False)
flag_summary.to_csv(flag_summary_path, index=False)

# -------------------------------------------------
# 9) Mostrar resultados
# -------------------------------------------------
print("\nResumen por estación:")
display(station_summary)

print("\nResumen de observaciones atípicas por estación:")
display(flag_summary)

print("\nPrimeras 20 observaciones sospechosas (IQR 1.5):")
display(flagged_rows.head(20))

print("\nArchivos guardados:")
print("-", station_summary_path)
print("-", duplicates_path)
print("-", flagged_rows_path)
print("-", flag_summary_path)

Resumen general del panel observado no2.5
- filas totales en archivo            : 862
- filas válidas no2.5                : 691
- estaciones con no2.5 válido        : 15
- meses observados no2.5             : 60
- rango temporal                     : 2020-01-01 a 2024-12-01

Duplicados por estación-mes:
- número de filas duplicadas: 0
- número de combinaciones duplicadas: 0

Resumen por estación:


,Estacion,n,mean,std,min,q01,q05,q25,median,q75,q95,q99,max
9,Movil Fontibon,41,22.487010,2.996244,14.659360,15.840765,17.617188,20.414468,22.826496,24.401067,27.140537,27.969767,28.411307
3,Colina,44,20.043809,4.408827,10.876454,11.824493,14.111747,16.633629,19.849463,23.248970,27.115494,28.245821,29.000852
10,Puente Aranda,55,19.414134,4.230511,10.513422,11.170235,12.407443,16.613826,19.111989,23.266281,25.886259,26.499810,26.837196
6,Kennedy,56,19.240490,2.686244,12.737240,13.130411,15.395727,17.780347,19.075312,20.965567,23.821699,24.812207,25.115036
4,Fontibon,56,18.527935,3.013420,10.848613,11.807391,12.858754,16.668824,19.073410,20.515434,22.516605,24.290668,25.339971
0,Bolivia,25,18.246689,2.074532,14.672035,14.688211,15.049291,17.192439,17.880621,19.100587,22.237043,23.091770,23.208772
7,Las Ferias,57,15.927606,3.767637,7.912392,8.664536,9.879546,13.215260,16.313752,18.476992,22.289905,23.175238,23.429806
8,MinAmbiente,41,15.059355,3.700623,7.839212,7.936239,8.568688,12.307660,15.644729,17.987408,19.929514,21.277854,21.476972
13,Tunal,51,13.668538,3.138572,7.492582,7.494033,9.579143,11.496209,13.813456,15.471360,19.056329,21.548852,22.522886
2,Centro de Alto Rendimiento,56,12.849231,2.904646,6.943706,7.075447,8.309460,10.365301,12.975367,14.834760,17.794590,18.458175,18.593842



Resumen de observaciones atípicas por estación:


,Estacion,n_total,n_flag_iqr15,n_flag_iqr30,NO2_mean,NO2_std,NO2_min,NO2_max,pct_flag_iqr15,pct_flag_iqr30
1,Carvajal - Sevillana,12,1,0,4.050724,1.160021,2.253492,6.711723,8.333333,0.0
0,Bolivia,25,2,0,18.246689,2.074532,14.672035,23.208772,8.000000,0.0
13,Tunal,51,1,0,13.668538,3.138572,7.492582,22.522886,1.960784,0.0
4,Fontibon,56,1,0,18.527935,3.013420,10.848613,25.339971,1.785714,0.0
6,Kennedy,56,1,0,19.240490,2.686244,12.737240,25.115036,1.785714,0.0
2,Centro de Alto Rendimiento,56,0,0,12.849231,2.904646,6.943706,18.593842,0.000000,0.0
3,Colina,44,0,0,20.043809,4.408827,10.876454,29.000852,0.000000,0.0
5,Guaymaral,54,0,0,9.928710,2.575733,5.090271,16.875758,0.000000,0.0
7,Las Ferias,57,0,0,15.927606,3.767637,7.912392,23.429806,0.000000,0.0
8,MinAmbiente,41,0,0,15.059355,3.700623,7.839212,21.476972,0.000000,0.0



Primeras 20 observaciones sospechosas (IQR 1.5):


,Estacion,fecha,Año,Mes,NO2_obs,Temp_media,HR,Vel_viento,Presión,Precipitación,q1,q3,iqr,lower_iqr15,upper_iqr15,flag_iqr15,flag_iqr30
171,Bolivia,2021-08-01,2021,8,22.721263,19.182903,73.339677,1.263871,NaN,71.11,17.192439,19.100587,1.908147,14.330219,21.962807,True,False
184,Bolivia,2021-09-01,2021,9,23.208772,19.143000,72.273333,1.281333,NaN,145.00,17.192439,19.100587,1.908147,14.330219,21.962807,True,False
68,Carvajal - Sevillana,2020-10-01,2020,10,6.711723,19.170968,72.356129,1.274516,NaN,60.73,3.237279,4.491524,1.254246,1.355910,6.372893,True,False
25,Fontibon,2020-04-01,2020,4,10.848613,19.681667,79.983333,0.890333,NaN,156.11,16.668824,20.515434,3.846610,10.898909,26.285350,True,False
449,Kennedy,2023-05-01,2023,5,12.737240,19.260323,81.586129,1.115484,NaN,220.91,17.780347,20.965567,3.185220,13.002517,25.743397,True,False
85,Tunal,2020-11-01,2020,11,22.522886,18.611333,81.529333,0.807667,NaN,71.11,11.496209,15.471360,3.975151,5.533482,21.434088,True,False



Archivos guardados:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_obs_audit_station_summary.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_obs_audit_duplicates.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_obs_audit_flagged_rows.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_obs_audit_flag_summary.csv


# CELDA 9A — Auditoría de calidad del panel observado no2.5

 **Objetivo:**
 revisar la calidad del panel observado de estaciones antes de hacer limpieza
 o volver a ajustar el HBM.

**Qué hace esta celda:**
 1. Lee el archivo corregido de estaciones.
 2. Construye la fecha mensual.
 3. Filtra observaciones válidas de no2.5.
 4. Revisa duplicados por estación y mes.
 5. Resume la distribución de no2.5 por estación:
    - número de observaciones
    - media
    - desviación estándar
    - percentiles
    - mínimo y máximo
 6. Marca observaciones atípicas por estación usando una regla robusta IQR.
 7. Genera una tabla con las filas sospechosas.
 8. Guarda archivos de auditoría para usarlos en la siguiente etapa de limpieza.

 **Importante:**
 esta celda no elimina datos.
 Solo identifica dónde puede estar entrando ruido al modelo.

In [12]:
# %%
# =========================
# CELDA 9B) Auditoría de covariables de malla no2.5
# =========================

grid_audit = pd.read_csv(GRID_CSV)
grid_audit["fecha"] = pd.to_datetime(grid_audit["fecha"], errors="coerce")

if grid_audit["fecha"].isna().any():
    raise ValueError("Hay fechas inválidas en grid_3km_AD_mensual_2020_2024.csv.")

covars_NO2 = [
    "Vel_viento_idw",
    "diff_NO2_grid",
    "grad_NO2_grid",
    "adv_proxy_NO2_grid",
]

missing_cols = [c for c in covars_NO2 if c not in grid_audit.columns]
if missing_cols:
    raise ValueError(f"Faltan columnas en la malla: {missing_cols}")

grid_audit["year"] = grid_audit["fecha"].dt.year
grid_audit["month"] = grid_audit["fecha"].dt.month

print("Resumen general de la malla auditada")
print("- filas totales       :", len(grid_audit))
print("- celdas únicas       :", grid_audit["cell_id"].nunique())
print("- meses únicos        :", grid_audit["fecha"].nunique())
print("- rango temporal      :", grid_audit["fecha"].min().date(), "a", grid_audit["fecha"].max().date())

# -------------------------------------------------
# 1) Resumen descriptivo de covariables
# -------------------------------------------------
desc_rows = []

for col in covars_NO2:
    s = pd.to_numeric(grid_audit[col], errors="coerce")
    desc_rows.append({
        "variable": col,
        "n": int(s.notna().sum()),
        "n_null": int(s.isna().sum()),
        "n_inf": int(np.isinf(s).sum()),
        "mean": float(np.nanmean(s)),
        "std": float(np.nanstd(s, ddof=1)),
        "min": float(np.nanmin(s)),
        "q01": float(np.nanquantile(s, 0.01)),
        "q05": float(np.nanquantile(s, 0.05)),
        "q25": float(np.nanquantile(s, 0.25)),
        "median": float(np.nanquantile(s, 0.50)),
        "q75": float(np.nanquantile(s, 0.75)),
        "q95": float(np.nanquantile(s, 0.95)),
        "q99": float(np.nanquantile(s, 0.99)),
        "max": float(np.nanmax(s)),
    })

desc_covars = pd.DataFrame(desc_rows)

# -------------------------------------------------
# 2) Flags IQR por covariable
# -------------------------------------------------
flag_rows = []
extreme_tables = []
monthly_flag_tables = []

for col in covars_NO2:
    tmp = grid_audit[["cell_id", "fecha", "year", "month", col]].copy()
    tmp[col] = pd.to_numeric(tmp[col], errors="coerce")

    q1 = tmp[col].quantile(0.25)
    q3 = tmp[col].quantile(0.75)
    iqr = q3 - q1

    lower_15 = q1 - 1.5 * iqr
    upper_15 = q3 + 1.5 * iqr
    lower_30 = q1 - 3.0 * iqr
    upper_30 = q3 + 3.0 * iqr

    tmp["variable"] = col
    tmp["q1"] = q1
    tmp["q3"] = q3
    tmp["iqr"] = iqr
    tmp["lower_iqr15"] = lower_15
    tmp["upper_iqr15"] = upper_15
    tmp["lower_iqr30"] = lower_30
    tmp["upper_iqr30"] = upper_30

    tmp["flag_iqr15"] = (tmp[col] < lower_15) | (tmp[col] > upper_15)
    tmp["flag_iqr30"] = (tmp[col] < lower_30) | (tmp[col] > upper_30)

    n_total = tmp[col].notna().sum()
    n_flag15 = int(tmp["flag_iqr15"].sum())
    n_flag30 = int(tmp["flag_iqr30"].sum())

    flag_rows.append({
        "variable": col,
        "n_total": int(n_total),
        "n_flag_iqr15": n_flag15,
        "n_flag_iqr30": n_flag30,
        "pct_flag_iqr15": 100 * n_flag15 / n_total if n_total else np.nan,
        "pct_flag_iqr30": 100 * n_flag30 / n_total if n_total else np.nan,
        "q1": float(q1),
        "q3": float(q3),
        "iqr": float(iqr),
        "lower_iqr15": float(lower_15),
        "upper_iqr15": float(upper_15),
        "lower_iqr30": float(lower_30),
        "upper_iqr30": float(upper_30),
    })

    # extremos por valor absoluto
    tmp["abs_value"] = tmp[col].abs()
    top_extreme = (
        tmp.sort_values("abs_value", ascending=False)
        .head(15)
        .copy()
    )
    extreme_tables.append(top_extreme)

    # resumen mensual de flags
    monthly_flags = (
        tmp.groupby(["year", "month"])
        .agg(
            n_total=(col, "count"),
            n_flag_iqr15=("flag_iqr15", "sum"),
            n_flag_iqr30=("flag_iqr30", "sum"),
            mean_value=(col, "mean"),
            std_value=(col, "std"),
        )
        .reset_index()
    )
    monthly_flags["variable"] = col
    monthly_flags["pct_flag_iqr15"] = 100 * monthly_flags["n_flag_iqr15"] / monthly_flags["n_total"]
    monthly_flags["pct_flag_iqr30"] = 100 * monthly_flags["n_flag_iqr30"] / monthly_flags["n_total"]
    monthly_flag_tables.append(monthly_flags)

flag_summary_covars = pd.DataFrame(flag_rows).sort_values("pct_flag_iqr15", ascending=False)
extreme_covars = pd.concat(extreme_tables, ignore_index=True)
monthly_flags_covars = pd.concat(monthly_flag_tables, ignore_index=True)

# -------------------------------------------------
# 3) Correlación entre covariables
# -------------------------------------------------
corr_covars = grid_audit[covars_NO2].corr(numeric_only=True)

# -------------------------------------------------
# 4) Guardar auditoría
# -------------------------------------------------
desc_covars_path = OUT_DIR / "HBM_NO2_grid_audit_covariate_summary.csv"
flag_covars_path = OUT_DIR / "HBM_NO2_grid_audit_flag_summary.csv"
extreme_covars_path = OUT_DIR / "HBM_NO2_grid_audit_extreme_rows.csv"
monthly_flags_covars_path = OUT_DIR / "HBM_NO2_grid_audit_monthly_flags.csv"
corr_covars_path = OUT_DIR / "HBM_NO2_grid_audit_correlation.csv"

desc_covars.to_csv(desc_covars_path, index=False)
flag_summary_covars.to_csv(flag_covars_path, index=False)
extreme_covars.to_csv(extreme_covars_path, index=False)
monthly_flags_covars.to_csv(monthly_flags_covars_path, index=False)
corr_covars.to_csv(corr_covars_path)

# -------------------------------------------------
# 5) Mostrar resultados
# -------------------------------------------------
print("\nResumen descriptivo de covariables:")
display(desc_covars)

print("\nResumen de flags por covariable:")
display(flag_summary_covars)

print("\nCorrelación entre covariables:")
display(corr_covars)

print("\nCasos más extremos por valor absoluto:")
display(
    extreme_covars[
        ["variable", "cell_id", "fecha", "year", "month"] + covars_NO2
    ].head(30)
)

print("\nMeses con mayor porcentaje de flags IQR 1.5 por covariable:")
top_months_flags = (
    monthly_flags_covars
    .sort_values(["variable", "pct_flag_iqr15"], ascending=[True, False])
    .groupby("variable")
    .head(10)
    .reset_index(drop=True)
)
display(top_months_flags)

print("\nArchivos guardados:")
print("-", desc_covars_path)
print("-", flag_covars_path)
print("-", extreme_covars_path)
print("-", monthly_flags_covars_path)
print("-", corr_covars_path)

Resumen general de la malla auditada
- filas totales       : 15240
- celdas únicas       : 254
- meses únicos        : 60
- rango temporal      : 2020-01-01 a 2024-12-01

Resumen descriptivo de covariables:


,variable,n,n_null,n_inf,mean,std,min,q01,q05,q25,median,q75,q95,q99,max
0,Vel_viento_idw,15240,0,0,1.116795e+00,0.221991,5.046784e-01,8.076667e-01,0.869237,0.944333,1.061290,1.215806,1.591290,1.717742,1.717742
1,diff_NO2_grid,15240,0,0,-3.788163e-18,4.235788,-4.650112e+01,-1.608892e+01,-2.885510,-0.171078,-0.003247,0.032364,3.905715,15.909122,33.887365
2,grad_NO2_grid,15240,0,0,1.141510e-04,0.000197,6.908695e-08,5.436184e-07,0.000001,0.000003,0.000018,0.000133,0.000540,0.000904,0.001837
3,adv_proxy_NO2_grid,15240,0,0,1.268041e-04,0.000224,1.016247e-07,5.281542e-07,0.000001,0.000004,0.000019,0.000147,0.000594,0.001037,0.002502



Resumen de flags por covariable:


,variable,n_total,n_flag_iqr15,n_flag_iqr30,pct_flag_iqr15,pct_flag_iqr30,q1,q3,iqr,lower_iqr15,upper_iqr15,lower_iqr30,upper_iqr30
1,diff_NO2_grid,15240,5442,4501,35.708661,29.534121,-0.171078,0.032364,0.203441,-0.476240,0.337526,-0.781402,0.642688
2,grad_NO2_grid,15240,1849,827,12.132546,5.426509,0.000003,0.000133,0.000129,-0.000191,0.000327,-0.000385,0.000521
3,adv_proxy_NO2_grid,15240,1801,822,11.817585,5.393701,0.000004,0.000147,0.000143,-0.000211,0.000361,-0.000425,0.000575
0,Vel_viento_idw,15240,500,0,3.280840,0.000000,0.944333,1.215806,0.271473,0.537124,1.623016,0.129914,2.030226



Correlación entre covariables:


...


Casos más extremos por valor absoluto:


,variable,cell_id,fecha,year,month,Vel_viento_idw,diff_NO2_grid,grad_NO2_grid,adv_proxy_NO2_grid
0,Vel_viento_idw,486,2023-07-01,2023,7,1.717742,NaN,NaN,NaN
1,Vel_viento_idw,464,2023-07-01,2023,7,1.717742,NaN,NaN,NaN
2,Vel_viento_idw,523,2023-07-01,2023,7,1.717742,NaN,NaN,NaN
3,Vel_viento_idw,304,2023-07-01,2023,7,1.717742,NaN,NaN,NaN
4,Vel_viento_idw,126,2023-07-01,2023,7,1.717742,NaN,NaN,NaN
5,Vel_viento_idw,169,2023-07-01,2023,7,1.717742,NaN,NaN,NaN
6,Vel_viento_idw,176,2023-07-01,2023,7,1.717742,NaN,NaN,NaN
7,Vel_viento_idw,2,2023-07-01,2023,7,1.717742,NaN,NaN,NaN
8,Vel_viento_idw,517,2023-07-01,2023,7,1.717742,NaN,NaN,NaN
9,Vel_viento_idw,519,2023-07-01,2023,7,1.717742,NaN,NaN,NaN



Meses con mayor porcentaje de flags IQR 1.5 por covariable:


,year,month,n_total,n_flag_iqr15,n_flag_iqr30,mean_value,std_value,variable,pct_flag_iqr15,pct_flag_iqr30
0,2023,7,254,254,0,1.713691e+00,0.014138,Vel_viento_idw,100.000000,0.000000
1,2021,7,254,241,0,1.637227e+00,0.006300,Vel_viento_idw,94.881890,0.000000
2,2020,11,254,5,0,7.889624e-01,0.059452,Vel_viento_idw,1.968504,0.000000
3,2020,1,254,0,0,1.480073e+00,0.073592,Vel_viento_idw,0.000000,0.000000
4,2020,2,254,0,0,1.350505e+00,0.035660,Vel_viento_idw,0.000000,0.000000
5,2020,3,254,0,0,1.037031e+00,0.042247,Vel_viento_idw,0.000000,0.000000
6,2020,4,254,0,0,8.910268e-01,0.002204,Vel_viento_idw,0.000000,0.000000
7,2020,5,254,0,0,1.143411e+00,0.002489,Vel_viento_idw,0.000000,0.000000
8,2020,6,254,0,0,1.243268e+00,0.002327,Vel_viento_idw,0.000000,0.000000
9,2020,7,254,0,0,1.134478e+00,0.010132,Vel_viento_idw,0.000000,0.000000



Archivos guardados:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_grid_audit_covariate_summary.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_grid_audit_flag_summary.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_grid_audit_extreme_rows.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_grid_audit_monthly_flags.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_grid_audit_correlation.csv



 # CELDA 9C — Construcción de covariables limpias para el HBM de no2.5

 **Objetivo:**
 crear una versión depurada y más estable de las covariables de malla
 para volver a correr el HBM con menor sensibilidad a colas extremas
 y menor redundancia entre predictores.

 **Estrategia aplicada:**
 - `Vel_viento_idw`: se conserva casi intacta
 - `diff_NO2_grid`: se transforma con signed-log y luego se winsoriza
 - `grad_NO2_grid`: se winsoriza solo para auditoría, pero no será usada
- `adv_proxy_NO2_grid`: se winsoriza y se conserva para el modelo

 **Variables finales recomendadas para el nuevo HBM:**
 - `Vel_viento_idw_clean`
 - `diff_NO2_grid_slog_clean`
 - `adv_proxy_NO2_grid_clean`

 **Importante:**
 en esta etapa no se reentrena el HBM.
 Solo se construye y guarda la malla limpia.

In [13]:
# %%
# =========================
# CELDA 9C) Construcción de covariables limpias no2.5
# =========================

grid_clean = grid_audit.copy()

def winsorize_series(s, q_low=0.01, q_high=0.99):
    s = pd.to_numeric(s, errors="coerce").astype(float)
    lo = s.quantile(q_low)
    hi = s.quantile(q_high)
    s_clip = s.clip(lower=lo, upper=hi)
    return s_clip, lo, hi

transform_report = []

# -------------------------------------------------
# 1) Velocidad del viento: conservar casi intacta
# -------------------------------------------------
grid_clean["Vel_viento_idw_clean"] = pd.to_numeric(grid_clean["Vel_viento_idw"], errors="coerce").astype(float)

transform_report.append({
    "variable_original": "Vel_viento_idw",
    "variable_limpia": "Vel_viento_idw_clean",
    "transformacion": "sin cambio",
    "q_low": np.nan,
    "q_high": np.nan
})

# -------------------------------------------------
# 2) diff_NO2_grid: signed-log + winsorización
# -------------------------------------------------
x_diff = pd.to_numeric(grid_clean["diff_NO2_grid"], errors="coerce").astype(float)
grid_clean["diff_NO2_grid_slog"] = np.sign(x_diff) * np.log1p(np.abs(x_diff))

grid_clean["diff_NO2_grid_slog_clean"], lo_diff, hi_diff = winsorize_series(
    grid_clean["diff_NO2_grid_slog"], q_low=0.01, q_high=0.99
)

transform_report.append({
    "variable_original": "diff_NO2_grid",
    "variable_limpia": "diff_NO2_grid_slog_clean",
    "transformacion": "signed_log1p + winsor_1_99",
    "q_low": float(lo_diff),
    "q_high": float(hi_diff)
})

# -------------------------------------------------
# 3) grad_NO2_grid: winsorización (solo auditoría / respaldo)
# -------------------------------------------------
grid_clean["grad_NO2_grid_clean"], lo_grad, hi_grad = winsorize_series(
    grid_clean["grad_NO2_grid"], q_low=0.01, q_high=0.99
)

transform_report.append({
    "variable_original": "grad_NO2_grid",
    "variable_limpia": "grad_NO2_grid_clean",
    "transformacion": "winsor_1_99",
    "q_low": float(lo_grad),
    "q_high": float(hi_grad)
})

# -------------------------------------------------
# 4) adv_proxy_NO2_grid: winsorización
# -------------------------------------------------
grid_clean["adv_proxy_NO2_grid_clean"], lo_adv, hi_adv = winsorize_series(
    grid_clean["adv_proxy_NO2_grid"], q_low=0.01, q_high=0.99
)

transform_report.append({
    "variable_original": "adv_proxy_NO2_grid",
    "variable_limpia": "adv_proxy_NO2_grid_clean",
    "transformacion": "winsor_1_99",
    "q_low": float(lo_adv),
    "q_high": float(hi_adv)
})

transform_report_df = pd.DataFrame(transform_report)

# -------------------------------------------------
# 5) Definir covariables recomendadas para el nuevo HBM
# -------------------------------------------------
X_COLS_NO2_CLEAN = [
    "Vel_viento_idw_clean",
    "diff_NO2_grid_slog_clean",
    "adv_proxy_NO2_grid_clean",
]

# -------------------------------------------------
# 6) Guardar archivo limpio
# -------------------------------------------------
grid_clean_path = OUT_DIR / "HBM_NO2_grid_clean_v1.csv"
transform_report_path = OUT_DIR / "HBM_NO2_grid_clean_transform_report.csv"
xcols_clean_path = OUT_DIR / "HBM_NO2_grid_clean_xcols.json"

grid_clean.to_csv(grid_clean_path, index=False)
transform_report_df.to_csv(transform_report_path, index=False)

with open(xcols_clean_path, "w", encoding="utf-8") as f:
    json.dump(X_COLS_NO2_CLEAN, f, ensure_ascii=False, indent=2)

# -------------------------------------------------
# 7) Resumen comparativo
# -------------------------------------------------
compare_summary = pd.DataFrame({
    "variable": [
        "Vel_viento_idw",
        "diff_NO2_grid",
        "diff_NO2_grid_slog_clean",
        "grad_NO2_grid",
        "grad_NO2_grid_clean",
        "adv_proxy_NO2_grid",
        "adv_proxy_NO2_grid_clean",
    ],
    "mean": [
        grid_clean["Vel_viento_idw"].mean(),
        grid_clean["diff_NO2_grid"].mean(),
        grid_clean["diff_NO2_grid_slog_clean"].mean(),
        grid_clean["grad_NO2_grid"].mean(),
        grid_clean["grad_NO2_grid_clean"].mean(),
        grid_clean["adv_proxy_NO2_grid"].mean(),
        grid_clean["adv_proxy_NO2_grid_clean"].mean(),
    ],
    "std": [
        grid_clean["Vel_viento_idw"].std(),
        grid_clean["diff_NO2_grid"].std(),
        grid_clean["diff_NO2_grid_slog_clean"].std(),
        grid_clean["grad_NO2_grid"].std(),
        grid_clean["grad_NO2_grid_clean"].std(),
        grid_clean["adv_proxy_NO2_grid"].std(),
        grid_clean["adv_proxy_NO2_grid_clean"].std(),
    ],
    "min": [
        grid_clean["Vel_viento_idw"].min(),
        grid_clean["diff_NO2_grid"].min(),
        grid_clean["diff_NO2_grid_slog_clean"].min(),
        grid_clean["grad_NO2_grid"].min(),
        grid_clean["grad_NO2_grid_clean"].min(),
        grid_clean["adv_proxy_NO2_grid"].min(),
        grid_clean["adv_proxy_NO2_grid_clean"].min(),
    ],
    "max": [
        grid_clean["Vel_viento_idw"].max(),
        grid_clean["diff_NO2_grid"].max(),
        grid_clean["diff_NO2_grid_slog_clean"].max(),
        grid_clean["grad_NO2_grid"].max(),
        grid_clean["grad_NO2_grid_clean"].max(),
        grid_clean["adv_proxy_NO2_grid"].max(),
        grid_clean["adv_proxy_NO2_grid_clean"].max(),
    ],
})

print("Covariables limpias construidas correctamente.\n")

print("Variables recomendadas para el nuevo HBM:")
print(X_COLS_NO2_CLEAN)

print("\nReporte de transformaciones:")
display(transform_report_df)

print("\nResumen comparativo antes/después:")
display(compare_summary)

print("\nArchivos guardados:")
print("-", grid_clean_path)
print("-", transform_report_path)
print("-", xcols_clean_path)

Covariables limpias construidas correctamente.

Variables recomendadas para el nuevo HBM:
['Vel_viento_idw_clean', 'diff_NO2_grid_slog_clean', 'adv_proxy_NO2_grid_clean']

Reporte de transformaciones:


,variable_original,variable_limpia,transformacion,q_low,q_high
0,Vel_viento_idw,Vel_viento_idw_clean,sin cambio,NaN,NaN
1,diff_NO2_grid,diff_NO2_grid_slog_clean,signed_log1p + winsor_1_99,-2.838430e+00,2.827853
2,grad_NO2_grid,grad_NO2_grid_clean,winsor_1_99,5.436184e-07,0.000904
3,adv_proxy_NO2_grid,adv_proxy_NO2_grid_clean,winsor_1_99,5.281542e-07,0.001037



Resumen comparativo antes/después:


,variable,mean,std,min,max
0,Vel_viento_idw,1.116795e+00,0.221991,5.046784e-01,1.717742
1,diff_NO2_grid,-3.788163e-18,4.235788,-4.650112e+01,33.887365
2,diff_NO2_grid_slog_clean,-1.915922e-02,0.843051,-2.838430e+00,2.827853
3,grad_NO2_grid,1.141510e-04,0.000197,6.908695e-08,0.001837
4,grad_NO2_grid_clean,1.120197e-04,0.000186,5.436184e-07,0.000904
5,adv_proxy_NO2_grid,1.268041e-04,0.000224,1.016247e-07,0.002502
6,adv_proxy_NO2_grid_clean,1.241771e-04,0.000210,5.281542e-07,0.001037



Archivos guardados:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_grid_clean_v1.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_grid_clean_transform_report.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_grid_clean_xcols.json



# CELDA 9D — Reconstrucción del panel HBM con covariables limpias

 **Objetivo:**
 construir la nueva base de modelación del HBM para no2.5 usando
 las covariables limpias generadas en la celda anterior.

 **Qué hace esta celda:**
 1. Toma la malla limpia `grid_clean`.
 2. Construye `grid_base_clean` con:
    - `cell_id`
    - `fecha`
    - `year`
    - `month`
    - `cell_idx`
    - `time_idx`
    - covariables limpias recomendadas
 3. Reconstruye el panel observado válido de no2.5.
 4. Une observaciones reales con la malla limpia por:
    - `cell_id`
    - `fecha`
 5. Guarda los archivos base para volver a correr el HBM limpio.

 **Salidas principales:**
 - `HBM_NO2_grid_base_clean_v1.csv`
 - `HBM_NO2_obs_panel_clean_v1.csv`
 - `HBM_NO2_clean_xcols.json`

In [14]:
# %%
# =========================
# CELDA 9D) Reconstruir panel HBM con covariables limpias
# =========================

# -------------------------------------------------
# 1) Verificaciones mínimas
# -------------------------------------------------
required_clean_cols = [
    "cell_id", "fecha",
    "Vel_viento_idw_clean",
    "diff_NO2_grid_slog_clean",
    "adv_proxy_NO2_grid_clean",
]

missing_clean = [c for c in required_clean_cols if c not in grid_clean.columns]
if missing_clean:
    raise ValueError(f"Faltan columnas en grid_clean: {missing_clean}")

# asegurar fecha
grid_clean["fecha"] = pd.to_datetime(grid_clean["fecha"], errors="coerce")
if grid_clean["fecha"].isna().any():
    raise ValueError("Hay fechas inválidas en grid_clean.")

grid_clean["year"] = grid_clean["fecha"].dt.year
grid_clean["month"] = grid_clean["fecha"].dt.month

# -------------------------------------------------
# 2) Construir grid_base_clean
# -------------------------------------------------
grid_base_clean = grid_clean[
    ["cell_id", "fecha", "year", "month"] + X_COLS_NO2_CLEAN
].copy()

# usar los mismos índices globales de celda y tiempo
grid_base_clean["cell_idx"] = grid_base_clean["cell_id"].map(cell_map)
grid_base_clean["time_idx"] = grid_base_clean["fecha"].map(time_map)

if grid_base_clean["cell_idx"].isna().any():
    raise ValueError("Hay cell_id en la malla limpia que no pudieron mapearse con cell_map.")
if grid_base_clean["time_idx"].isna().any():
    raise ValueError("Hay fechas en la malla limpia que no pudieron mapearse con time_map.")

grid_base_clean["cell_idx"] = grid_base_clean["cell_idx"].astype(int)
grid_base_clean["time_idx"] = grid_base_clean["time_idx"].astype(int)

# -------------------------------------------------
# 3) Reconstruir observaciones válidas no2.5
# -------------------------------------------------
obs_clean_src = pd.read_csv(OBS_CSV)

obs_clean_src["fecha"] = pd.to_datetime(
    dict(year=obs_clean_src["Año"].astype(int), month=obs_clean_src["Mes"].astype(int), day=1),
    errors="coerce"
)

if obs_clean_src["fecha"].isna().any():
    raise ValueError("Hay fechas inválidas en el panel observado.")

obs_clean_src["valido_NO2"] = obs_clean_src["valido_NO2"].astype(bool)

obs_no_clean = obs_clean_src.loc[
    (obs_clean_src["valido_NO2"] == True) &
    (obs_clean_src["NO2"].notna()) &
    (obs_clean_src["cell_id"].notna())
].copy()

obs_no_clean = obs_no_clean.rename(columns={"NO2": "NO2_obs"})

# -------------------------------------------------
# 4) Unir observaciones con covariables limpias
# -------------------------------------------------
obs_panel_clean = obs_no_clean.merge(
    grid_base_clean,
    on=["cell_id", "fecha"],
    how="left",
    validate="many_to_one"
)

missing_covs_clean = obs_panel_clean[X_COLS_NO2_CLEAN].isna().any(axis=1).sum()
if missing_covs_clean > 0:
    raise ValueError(
        f"Hay {missing_covs_clean} observaciones sin covariables limpias. "
        "Revisa cell_id y fecha."
    )

# -------------------------------------------------
# 5) Guardar archivos base limpios
# -------------------------------------------------
grid_base_clean_path = OUT_DIR / "HBM_NO2_grid_base_clean_v1.csv"
obs_panel_clean_path = OUT_DIR / "HBM_NO2_obs_panel_clean_v1.csv"
xcols_clean_path = OUT_DIR / "HBM_NO2_clean_xcols.json"

grid_base_clean.to_csv(grid_base_clean_path, index=False)
obs_panel_clean.to_csv(obs_panel_clean_path, index=False)

with open(xcols_clean_path, "w", encoding="utf-8") as f:
    json.dump(X_COLS_NO2_CLEAN, f, ensure_ascii=False, indent=2)

# -------------------------------------------------
# 6) Resumen
# -------------------------------------------------
print("Panel HBM limpio reconstruido correctamente.\n")

print("grid_base_clean:")
print("- filas            :", len(grid_base_clean))
print("- celdas únicas    :", grid_base_clean["cell_id"].nunique())
print("- meses únicos     :", grid_base_clean["fecha"].nunique())

print("\nobs_panel_clean:")
print("- filas            :", len(obs_panel_clean))
print("- estaciones únicas:", obs_panel_clean["Estacion"].nunique())
print("- celdas observadas:", obs_panel_clean["cell_id"].nunique())

print("\nCovariables limpias usadas:")
print(X_COLS_NO2_CLEAN)

print("\nArchivos guardados:")
print("-", grid_base_clean_path)
print("-", obs_panel_clean_path)
print("-", xcols_clean_path)

print("\nPrimeras filas de obs_panel_clean:")
display(
    obs_panel_clean[
        ["Estacion", "fecha", "NO2_obs", "cell_id"] + X_COLS_NO2_CLEAN
    ].head()
)

Panel HBM limpio reconstruido correctamente.

grid_base_clean:
- filas            : 15240
- celdas únicas    : 254
- meses únicos     : 60

obs_panel_clean:
- filas            : 691
- estaciones únicas: 15
- celdas observadas: 13

Covariables limpias usadas:
['Vel_viento_idw_clean', 'diff_NO2_grid_slog_clean', 'adv_proxy_NO2_grid_clean']

Archivos guardados:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_grid_base_clean_v1.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_obs_panel_clean_v1.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_clean_xcols.json

Primeras filas de obs_panel_clean:


,Estacion,fecha,NO2_obs,cell_id,Vel_viento_idw_clean,diff_NO2_grid_slog_clean,adv_proxy_NO2_grid_clean
0,Carvajal - Sevillana,2020-01-01,3.291089,482,1.503226,2.174639,0.000601
1,Centro de Alto Rendimiento,2020-01-01,13.822458,567,1.503226,2.285265,0.000464
2,Fontibon,2020-01-01,16.696307,485,1.503226,-1.478700,0.000214
3,Guaymaral,2020-01-01,11.346686,653,1.128177,1.865474,0.000227
4,Kennedy,2020-01-01,17.867042,442,1.503226,-2.361676,0.001037



# CELDA 9E — Validación temporal del HBM limpio (M1b-clean)

 **Objetivo:**
 volver a correr la validación temporal del HBM usando la versión limpia
 de las covariables de malla para no2.5.

 **Modelo usado:**
 - estructura: `M1b`
 - espacial: `ICAR`
 - temporal: `RW1`
 - observación: `NO2_obs`

 **Covariables limpias:**
 - `Vel_viento_idw_clean`
 - `diff_NO2_grid_slog_clean`
 - `adv_proxy_NO2_grid_clean`

 **Qué hace esta celda:**
 1. Usa `grid_base_clean` y `obs_panel_clean`.
 2. Repite los 3 folds temporales.
 3. Escala covariables usando solo train en cada fold.
 4. Ajusta `fit_hbm_m1b_fold`.
 5. Guarda predicciones, summary y parámetros de escalamiento.
 6. Consolida métricas.
 7. Compara contra el modelo previo `M1b`.

 **Salidas principales:**
 - `HBM_NO2_M1bclean_fold_1_pred_test.csv`
 - `HBM_NO2_M1bclean_fold_2_pred_test.csv`
 - `HBM_NO2_M1bclean_fold_3_pred_test.csv`
 - `HBM_NO2_M1bclean_metrics_folds.csv`
 - `HBM_NO2_compare_M1b_vs_M1bclean.csv`

 **Importante:**
 aquí lo que más nos interesa revisar después es si bajan:
 - `width_90_mean`
 - `wis_90`
 sin deteriorar demasiado MAE/RMSE.

In [15]:
# %%
# =========================
# CELDA 9E) Validación temporal HBM limpio
# =========================

all_metrics_m1bclean = []
all_test_preds_m1bclean = []

DRAWS_C = 1000
TUNE_C = 1500
CHAINS_C = 4
TARGET_ACCEPT_C = 0.99
MAX_TREEDEPTH_C = 15
RANDOM_SEED_C = 42

for fold_name, fold_info in FOLDS.items():
    print("\n" + "=" * 80)
    print(f"Corriendo M1b-clean - {fold_name}")
    print("Train years:", fold_info["train_years"])
    print("Test years :", fold_info["test_years"])

    # -------------------------------------------------
    # 1) Escalar covariables limpias usando solo train
    # -------------------------------------------------
    grid_scaled_c, scale_params_c = scale_grid_by_train_years(
        grid_df=grid_base_clean,
        x_cols=X_COLS_NO2_CLEAN,
        train_years=fold_info["train_years"]
    )

    # -------------------------------------------------
    # 2) Llevar covariables escaladas al panel observado limpio
    # -------------------------------------------------
    obs_scaled_c = merge_scaled_covariates_to_obs(
        obs_df=obs_panel_clean,
        grid_scaled=grid_scaled_c,
        x_cols=X_COLS_NO2_CLEAN
    )

    # -------------------------------------------------
    # 3) Ajustar modelo del fold
    # -------------------------------------------------
    idata_fold_c, test_pred_fold_c, met_fold_c = fit_hbm_m1b_fold(
        obs_scaled=obs_scaled_c,
        grid_scaled=grid_scaled_c,
        w_edges_df=w_edges,
        x_cols_raw=X_COLS_NO2_CLEAN,
        train_years=fold_info["train_years"],
        test_years=fold_info["test_years"],
        n_cells=len(cell_ids),
        n_times=len(time_ids),
        draws=DRAWS_C,
        tune=TUNE_C,
        chains=CHAINS_C,
        target_accept=TARGET_ACCEPT_C,
        max_treedepth=MAX_TREEDEPTH_C,
        random_seed=RANDOM_SEED_C,
    )

    # -------------------------------------------------
    # 4) Guardar predicciones del fold
    # -------------------------------------------------
    test_pred_fold_c["fold"] = fold_name
    pred_fold_path_c = OUT_DIR / f"HBM_NO2_M1bclean_{fold_name}_pred_test.csv"
    test_pred_fold_c.to_csv(pred_fold_path_c, index=False)

    # -------------------------------------------------
    # 5) Guardar summary del posterior
    # -------------------------------------------------
    summary_fold_c = az.summary(
        idata_fold_c,
        var_names=["alpha", "beta", "sigma_y", "tau_phi", "sigma_t"],
        round_to=4
    )
    summary_fold_path_c = OUT_DIR / f"HBM_NO2_M1bclean_{fold_name}_summary.csv"
    summary_fold_c.to_csv(summary_fold_path_c)

    # -------------------------------------------------
    # 6) Guardar parámetros de escalamiento
    # -------------------------------------------------
    scale_fold_path_c = OUT_DIR / f"HBM_NO2_M1bclean_{fold_name}_scale_params.json"
    with open(scale_fold_path_c, "w", encoding="utf-8") as f:
        json.dump(scale_params_c, f, ensure_ascii=False, indent=2)

    # -------------------------------------------------
    # 7) Consolidar métricas
    # -------------------------------------------------
    met_fold_c["model"] = "M1b_clean"
    met_fold_c["fold"] = fold_name
    met_fold_c["train_years"] = ",".join(map(str, fold_info["train_years"]))
    met_fold_c["test_years"] = ",".join(map(str, fold_info["test_years"]))

    all_metrics_m1bclean.append(met_fold_c)
    all_test_preds_m1bclean.append(test_pred_fold_c)

    print("\nGuardado del fold:")
    print("-", pred_fold_path_c)
    print("-", summary_fold_path_c)
    print("-", scale_fold_path_c)

    print("\nMétricas del fold:")
    for k, v in met_fold_c.items():
        print(f"- {k}: {v}")

# -------------------------------------------------
# 8) Consolidar resultados M1b-clean
# -------------------------------------------------
metrics_m1bclean_df = pd.DataFrame(all_metrics_m1bclean)
preds_m1bclean_df = pd.concat(all_test_preds_m1bclean, ignore_index=True)

metrics_m1bclean_path = OUT_DIR / "HBM_NO2_M1bclean_metrics_folds.csv"
preds_m1bclean_path = OUT_DIR / "HBM_NO2_M1bclean_pred_test_all_folds.csv"

metrics_m1bclean_df.to_csv(metrics_m1bclean_path, index=False)
preds_m1bclean_df.to_csv(preds_m1bclean_path, index=False)

print("\n" + "=" * 80)
print("VALIDACIÓN TEMPORAL M1b-clean COMPLETADA")
print("- métricas consolidadas :", metrics_m1bclean_path)
print("- predicciones consolidadas :", preds_m1bclean_path)

print("\nResumen final de métricas M1b-clean:")
display(metrics_m1bclean_df)

# -------------------------------------------------
# 9) Comparar M1b vs M1b-clean
# -------------------------------------------------
m1b_metrics_path = OUT_DIR / "HBM_NO2_M1b_metrics_folds.csv"

if m1b_metrics_path.exists():
    metrics_m1b_df = pd.read_csv(m1b_metrics_path).copy()
    metrics_m1b_df["model"] = "M1b"

    cols_keep = [
        "model", "fold", "n", "mae", "rmse", "bias", "r", "r2",
        "coverage_90", "width_90_mean", "wis_90", "train_years", "test_years"
    ]

    compare_clean_df = pd.concat(
        [
            metrics_m1b_df[cols_keep],
            metrics_m1bclean_df[cols_keep]
        ],
        ignore_index=True
    )

    compare_clean_path = OUT_DIR / "HBM_NO2_compare_M1b_vs_M1bclean.csv"
    compare_clean_df.to_csv(compare_clean_path, index=False)

    print("\nComparación M1b vs M1b-clean guardada en:")
    print("-", compare_clean_path)

    print("\nTabla comparativa:")
    display(compare_clean_df.sort_values(["fold", "model"]).reset_index(drop=True))
else:
    print("\nNo se encontró el archivo de métricas de M1b previo.")
    print("Se omitió la comparación M1b vs M1b-clean.")


Corriendo M1b-clean - fold_1
Train years: [2020, 2021]
Test years : [2022]


Initializing NUTS using adapt_diag...
ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: local_subtensor_merge
ERROR (pytensor.graph.rewriting.basic): node: Subtensor{i}(Subtensor{start:}.0, 0)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 1920, in process_node
    replacements = node_rewriter.transform(
                   ^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 993, in transform
    return self.fn(fgraph, node)
           ^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 416, in local_subtensor_merge
    merge_two_slices(
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 865, in

Output()

Sampling 4 chains for 1_500 tune and 1_000 draw iterations (6_000 + 4_000 draws total) took 402 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



Guardado del fold:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1bclean_fold_1_pred_test.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1bclean_fold_1_summary.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1bclean_fold_1_scale_params.json

Métricas del fold:
- n: 155
- mae: 2.654196317904939
- rmse: 3.2570956720803657
- bias: -0.3682381446306532
- r: 0.7459634657254368
- r2: 0.556461492197105
- coverage_90: 0.9870967741935484
- width_90_mean: 20.93448742056576
- wis_90: 21.55697901827244
- model: M1b_clean
- fold: fold_1
- train_years: 2020,2021
- test_years: 2022

Corriendo M1b-clean - fold_2
Train years: [2020, 2021, 2022]
Test years : [2023]


...

Sampling 4 chains for 1_500 tune and 1_000 draw iterations (6_000 + 4_000 draws total) took 1447 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



Guardado del fold:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1bclean_fold_3_pred_test.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1bclean_fold_3_summary.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1bclean_fold_3_scale_params.json

Métricas del fold:
- n: 144
- mae: 2.38712045669478
- rmse: 2.9585864718066626
- bias: 0.7680251265189325
- r: 0.8372888770956016
- r2: 0.7010526637080134
- coverage_90: 0.9583333333333334
- width_90_mean: 18.684243397609443
- wis_90: 19.288280033283527
- model: M1b_clean
- fold: fold_3
- train_years: 2020,2021,2022,2023
- test_years: 2024

VALIDACIÓN TEMPORAL M1b-clean COMPLETADA
- métricas consolidadas : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1bclean_metrics_folds.csv
- predicciones consolidadas : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1bclean_pred_test_all_folds.csv

Resumen final de métricas M1b-clean:


,n,mae,rmse,bias,r,r2,coverage_90,width_90_mean,wis_90,model,fold,train_years,test_years
0,155,2.654196,3.257096,-0.368238,0.745963,0.556461,0.987097,20.934487,21.556979,M1b_clean,fold_1,"2020,2021",2022
1,154,2.255073,2.757524,0.553443,0.860652,0.740722,0.987013,20.270887,20.380787,M1b_clean,fold_2,"2020,2021,2022",2023
2,144,2.387120,2.958586,0.768025,0.837289,0.701053,0.958333,18.684243,19.288280,M1b_clean,fold_3,"2020,2021,2022,2023",2024



Comparación M1b vs M1b-clean guardada en:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_compare_M1b_vs_M1bclean.csv

Tabla comparativa:


,model,fold,n,mae,rmse,bias,r,r2,coverage_90,width_90_mean,wis_90,train_years,test_years
0,M1b,fold_1,155,2.506299,3.083538,-0.204953,0.775185,0.600911,0.987097,21.723954,22.286719,"2020,2021",2022
1,M1b_clean,fold_1,155,2.654196,3.257096,-0.368238,0.745963,0.556461,0.987097,20.934487,21.556979,"2020,2021",2022
2,M1b,fold_2,154,2.322275,2.810962,0.868654,0.862314,0.743585,0.974026,20.609334,20.753718,"2020,2021,2022",2023
3,M1b_clean,fold_2,154,2.255073,2.757524,0.553443,0.860652,0.740722,0.987013,20.270887,20.380787,"2020,2021,2022",2023
4,M1b,fold_3,144,2.346842,2.880061,0.846739,0.851661,0.725326,0.972222,19.332074,19.771937,"2020,2021,2022,2023",2024
5,M1b_clean,fold_3,144,2.387120,2.958586,0.768025,0.837289,0.701053,0.958333,18.684243,19.288280,"2020,2021,2022,2023",2024


In [16]:
# %%
# =========================
# CELDA 9E) Validación temporal HBM limpio
# =========================

all_metrics_m1bclean = []
all_test_preds_m1bclean = []

DRAWS_C = 1000
TUNE_C = 1500
CHAINS_C = 4
TARGET_ACCEPT_C = 0.99
MAX_TREEDEPTH_C = 15
RANDOM_SEED_C = 42

for fold_name, fold_info in FOLDS.items():
    print("\n" + "=" * 80)
    print(f"Corriendo M1b-clean - {fold_name}")
    print("Train years:", fold_info["train_years"])
    print("Test years :", fold_info["test_years"])

    # -------------------------------------------------
    # 1) Escalar covariables limpias usando solo train
    # -------------------------------------------------
    grid_scaled_c, scale_params_c = scale_grid_by_train_years(
        grid_df=grid_base_clean,
        x_cols=X_COLS_NO2_CLEAN,
        train_years=fold_info["train_years"]
    )

    # -------------------------------------------------
    # 2) Llevar covariables escaladas al panel observado limpio
    # -------------------------------------------------
    obs_scaled_c = merge_scaled_covariates_to_obs(
        obs_df=obs_panel_clean,
        grid_scaled=grid_scaled_c,
        x_cols=X_COLS_NO2_CLEAN
    )

    # -------------------------------------------------
    # 3) Ajustar modelo del fold
    # -------------------------------------------------
    idata_fold_c, test_pred_fold_c, met_fold_c = fit_hbm_m1b_fold(
        obs_scaled=obs_scaled_c,
        grid_scaled=grid_scaled_c,
        w_edges_df=w_edges,
        x_cols_raw=X_COLS_NO2_CLEAN,
        train_years=fold_info["train_years"],
        test_years=fold_info["test_years"],
        n_cells=len(cell_ids),
        n_times=len(time_ids),
        draws=DRAWS_C,
        tune=TUNE_C,
        chains=CHAINS_C,
        target_accept=TARGET_ACCEPT_C,
        max_treedepth=MAX_TREEDEPTH_C,
        random_seed=RANDOM_SEED_C,
    )

    # -------------------------------------------------
    # 4) Guardar predicciones del fold
    # -------------------------------------------------
    test_pred_fold_c["fold"] = fold_name
    pred_fold_path_c = OUT_DIR / f"HBM_NO2_M1bclean_{fold_name}_pred_test.csv"
    test_pred_fold_c.to_csv(pred_fold_path_c, index=False)

    # -------------------------------------------------
    # 5) Guardar summary del posterior
    # -------------------------------------------------
    summary_fold_c = az.summary(
        idata_fold_c,
        var_names=["alpha", "beta", "sigma_y", "tau_phi", "sigma_t"],
        round_to=4
    )
    summary_fold_path_c = OUT_DIR / f"HBM_NO2_M1bclean_{fold_name}_summary.csv"
    summary_fold_c.to_csv(summary_fold_path_c)

    # -------------------------------------------------
    # 6) Guardar parámetros de escalamiento
    # -------------------------------------------------
    scale_fold_path_c = OUT_DIR / f"HBM_NO2_M1bclean_{fold_name}_scale_params.json"
    with open(scale_fold_path_c, "w", encoding="utf-8") as f:
        json.dump(scale_params_c, f, ensure_ascii=False, indent=2)

    # -------------------------------------------------
    # 7) Consolidar métricas
    # -------------------------------------------------
    met_fold_c["model"] = "M1b_clean"
    met_fold_c["fold"] = fold_name
    met_fold_c["train_years"] = ",".join(map(str, fold_info["train_years"]))
    met_fold_c["test_years"] = ",".join(map(str, fold_info["test_years"]))

    all_metrics_m1bclean.append(met_fold_c)
    all_test_preds_m1bclean.append(test_pred_fold_c)

    print("\nGuardado del fold:")
    print("-", pred_fold_path_c)
    print("-", summary_fold_path_c)
    print("-", scale_fold_path_c)

    print("\nMétricas del fold:")
    for k, v in met_fold_c.items():
        print(f"- {k}: {v}")

# -------------------------------------------------
# 8) Consolidar resultados M1b-clean
# -------------------------------------------------
metrics_m1bclean_df = pd.DataFrame(all_metrics_m1bclean)
preds_m1bclean_df = pd.concat(all_test_preds_m1bclean, ignore_index=True)

metrics_m1bclean_path = OUT_DIR / "HBM_NO2_M1bclean_metrics_folds.csv"
preds_m1bclean_path = OUT_DIR / "HBM_NO2_M1bclean_pred_test_all_folds.csv"

metrics_m1bclean_df.to_csv(metrics_m1bclean_path, index=False)
preds_m1bclean_df.to_csv(preds_m1bclean_path, index=False)

print("\n" + "=" * 80)
print("VALIDACIÓN TEMPORAL M1b-clean COMPLETADA")
print("- métricas consolidadas :", metrics_m1bclean_path)
print("- predicciones consolidadas :", preds_m1bclean_path)

print("\nResumen final de métricas M1b-clean:")
display(metrics_m1bclean_df)

# -------------------------------------------------
# 9) Comparar M1b vs M1b-clean
# -------------------------------------------------
m1b_metrics_path = OUT_DIR / "HBM_NO2_M1b_metrics_folds.csv"

if m1b_metrics_path.exists():
    metrics_m1b_df = pd.read_csv(m1b_metrics_path).copy()
    metrics_m1b_df["model"] = "M1b"

    cols_keep = [
        "model", "fold", "n", "mae", "rmse", "bias", "r", "r2",
        "coverage_90", "width_90_mean", "wis_90", "train_years", "test_years"
    ]

    compare_clean_df = pd.concat(
        [
            metrics_m1b_df[cols_keep],
            metrics_m1bclean_df[cols_keep]
        ],
        ignore_index=True
    )

    compare_clean_path = OUT_DIR / "HBM_NO2_compare_M1b_vs_M1bclean.csv"
    compare_clean_df.to_csv(compare_clean_path, index=False)

    print("\nComparación M1b vs M1b-clean guardada en:")
    print("-", compare_clean_path)

    print("\nTabla comparativa:")
    display(compare_clean_df.sort_values(["fold", "model"]).reset_index(drop=True))
else:
    print("\nNo se encontró el archivo de métricas de M1b previo.")
    print("Se omitió la comparación M1b vs M1b-clean.")


Corriendo M1b-clean - fold_1
Train years: [2020, 2021]
Test years : [2022]


Initializing NUTS using adapt_diag...
ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: local_subtensor_merge
ERROR (pytensor.graph.rewriting.basic): node: Subtensor{i}(Subtensor{start:}.0, 0)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 1920, in process_node
    replacements = node_rewriter.transform(
                   ^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 993, in transform
    return self.fn(fgraph, node)
           ^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 416, in local_subtensor_merge
    merge_two_slices(
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 865, in

Output()

Sampling 4 chains for 1_500 tune and 1_000 draw iterations (6_000 + 4_000 draws total) took 432 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



Guardado del fold:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1bclean_fold_1_pred_test.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1bclean_fold_1_summary.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1bclean_fold_1_scale_params.json

Métricas del fold:
- n: 155
- mae: 2.654196317904939
- rmse: 3.2570956720803657
- bias: -0.3682381446306532
- r: 0.7459634657254368
- r2: 0.556461492197105
- coverage_90: 0.9870967741935484
- width_90_mean: 20.93448742056576
- wis_90: 21.55697901827244
- model: M1b_clean
- fold: fold_1
- train_years: 2020,2021
- test_years: 2022

Corriendo M1b-clean - fold_2
Train years: [2020, 2021, 2022]
Test years : [2023]


...

Sampling 4 chains for 1_500 tune and 1_000 draw iterations (6_000 + 4_000 draws total) took 1565 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



Guardado del fold:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1bclean_fold_3_pred_test.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1bclean_fold_3_summary.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1bclean_fold_3_scale_params.json

Métricas del fold:
- n: 144
- mae: 2.38712045669478
- rmse: 2.9585864718066626
- bias: 0.7680251265189325
- r: 0.8372888770956016
- r2: 0.7010526637080134
- coverage_90: 0.9583333333333334
- width_90_mean: 18.684243397609443
- wis_90: 19.288280033283527
- model: M1b_clean
- fold: fold_3
- train_years: 2020,2021,2022,2023
- test_years: 2024

VALIDACIÓN TEMPORAL M1b-clean COMPLETADA
- métricas consolidadas : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1bclean_metrics_folds.csv
- predicciones consolidadas : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1bclean_pred_test_all_folds.csv

Resumen final de métricas M1b-clean:


,n,mae,rmse,bias,r,r2,coverage_90,width_90_mean,wis_90,model,fold,train_years,test_years
0,155,2.654196,3.257096,-0.368238,0.745963,0.556461,0.987097,20.934487,21.556979,M1b_clean,fold_1,"2020,2021",2022
1,154,2.255073,2.757524,0.553443,0.860652,0.740722,0.987013,20.270887,20.380787,M1b_clean,fold_2,"2020,2021,2022",2023
2,144,2.387120,2.958586,0.768025,0.837289,0.701053,0.958333,18.684243,19.288280,M1b_clean,fold_3,"2020,2021,2022,2023",2024



Comparación M1b vs M1b-clean guardada en:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_compare_M1b_vs_M1bclean.csv

Tabla comparativa:


,model,fold,n,mae,rmse,bias,r,r2,coverage_90,width_90_mean,wis_90,train_years,test_years
0,M1b,fold_1,155,2.506299,3.083538,-0.204953,0.775185,0.600911,0.987097,21.723954,22.286719,"2020,2021",2022
1,M1b_clean,fold_1,155,2.654196,3.257096,-0.368238,0.745963,0.556461,0.987097,20.934487,21.556979,"2020,2021",2022
2,M1b,fold_2,154,2.322275,2.810962,0.868654,0.862314,0.743585,0.974026,20.609334,20.753718,"2020,2021,2022",2023
3,M1b_clean,fold_2,154,2.255073,2.757524,0.553443,0.860652,0.740722,0.987013,20.270887,20.380787,"2020,2021,2022",2023
4,M1b,fold_3,144,2.346842,2.880061,0.846739,0.851661,0.725326,0.972222,19.332074,19.771937,"2020,2021,2022,2023",2024
5,M1b_clean,fold_3,144,2.387120,2.958586,0.768025,0.837289,0.701053,0.958333,18.684243,19.288280,"2020,2021,2022,2023",2024


# CELDA 9F — Comparación de diagnósticos de convergencia: M1b vs M1b-clean

 **Objetivo:**
 verificar si la limpieza de covariables no solo mejoró las métricas predictivas,
 sino también la estabilidad bayesiana del muestreo.

 **Qué hace esta celda:**
 1. Lee los archivos `summary.csv` de:
    - `M1b`
    - `M1b_clean`
 2. Extrae por fold:
    - `max_r_hat`
    - `min_ess_bulk`
    - `min_ess_tail`
 3. Consolida la comparación.
 4. Marca reglas simples:
    - `r_hat_ok`: max r_hat <= 1.01
    - `ess_bulk_ok`: min ess_bulk >= 400
    - `ess_tail_ok`: min ess_tail >= 400

 **Interpretación esperada:**
 si `M1b_clean` mantiene o mejora estos diagnósticos, será el modelo elegido
 para el ajuste final sobre todo el período.

In [17]:
# %%
# =========================
# CELDA 9F) Diagnósticos M1b vs M1b-clean
# =========================

summary_files_clean_compare = {
    "M1b": {
        "fold_1": OUT_DIR / "HBM_NO2_M1b_fold_1_summary.csv",
        "fold_2": OUT_DIR / "HBM_NO2_M1b_fold_2_summary.csv",
        "fold_3": OUT_DIR / "HBM_NO2_M1b_fold_3_summary.csv",
    },
    "M1b_clean": {
        "fold_1": OUT_DIR / "HBM_NO2_M1bclean_fold_1_summary.csv",
        "fold_2": OUT_DIR / "HBM_NO2_M1bclean_fold_2_summary.csv",
        "fold_3": OUT_DIR / "HBM_NO2_M1bclean_fold_3_summary.csv",
    }
}

rows_diag_clean = []

for model_name, model_files in summary_files_clean_compare.items():
    for fold_name, path_summary in model_files.items():
        if not path_summary.exists():
            print(f"No existe: {path_summary}")
            continue

        df_sum = pd.read_csv(path_summary, index_col=0)

        needed_cols = ["r_hat", "ess_bulk", "ess_tail"]
        for c in needed_cols:
            if c not in df_sum.columns:
                raise ValueError(f"Falta la columna '{c}' en {path_summary.name}")

        row = {
            "model": model_name,
            "fold": fold_name,
            "max_r_hat": float(df_sum["r_hat"].max()),
            "min_ess_bulk": float(df_sum["ess_bulk"].min()),
            "min_ess_tail": float(df_sum["ess_tail"].min()),
        }

        row["r_hat_ok"] = row["max_r_hat"] <= 1.01
        row["ess_bulk_ok"] = row["min_ess_bulk"] >= 400
        row["ess_tail_ok"] = row["min_ess_tail"] >= 400

        rows_diag_clean.append(row)

diag_clean_compare = (
    pd.DataFrame(rows_diag_clean)
    .sort_values(["fold", "model"])
    .reset_index(drop=True)
)

diag_clean_compare_path = OUT_DIR / "HBM_NO2_compare_diagnostics_M1b_vs_M1bclean.csv"
diag_clean_compare.to_csv(diag_clean_compare_path, index=False)

print("Diagnósticos comparativos guardados en:")
print("-", diag_clean_compare_path)

print("\nTabla comparativa de convergencia:")
display(diag_clean_compare)

Diagnósticos comparativos guardados en:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_compare_diagnostics_M1b_vs_M1bclean.csv

Tabla comparativa de convergencia:


,model,fold,max_r_hat,min_ess_bulk,min_ess_tail,r_hat_ok,ess_bulk_ok,ess_tail_ok
0,M1b,fold_1,1.0034,501.6503,925.9753,True,True,True
1,M1b_clean,fold_1,1.0038,631.0893,1042.3280,True,True,True
2,M1b,fold_2,1.0021,472.5702,1150.8206,True,True,True
3,M1b_clean,fold_2,1.0091,516.4569,1440.7685,True,True,True
4,M1b,fold_3,1.0023,546.4780,943.2620,True,True,True
5,M1b_clean,fold_3,1.0130,517.1691,1079.9292,False,True,True


# CELDA 10 — Ajuste final del modelo seleccionado M1b-clean

 **Objetivo:**
 ajustar el modelo final seleccionado para no2.5 usando todas las
 observaciones válidas de 2020–2024 y las covariables limpias.

 **Modelo final seleccionado:**
 - observación: `NO2_obs`
 - espacial: `ICAR`
 - temporal: `RW1`
 - covariables limpias:
 - `Vel_viento_idw_clean`
 - `diff_NO2_grid_slog_clean`
 - `adv_proxy_NO2_grid_clean`

 **Qué hace esta celda:**
 1. Escala covariables limpias usando todo el período.
 2. Ajusta el modelo final `M1b_clean`.
 3. Predice sobre toda la malla 3 km y todos los meses.
 4. Exporta la superficie final:
    - `p05_hbm`
    - `p50_hbm`
    - `p95_hbm`
    - `width_90`
 5. Guarda también:
    - summary final
    - diagnósticos finales
    - parámetros de escalamiento
    - posterior en NetCDF

 **Salidas principales:**
 - `HBM_NO2_M1bclean_surface_final.csv`
 - `HBM_NO2_M1bclean_final_summary.csv`
 - `HBM_NO2_M1bclean_final_diagnostics.csv`
 - `HBM_NO2_M1bclean_final_scale_params.json`
 - `HBM_NO2_M1bclean_final_posterior.nc`

In [18]:
# %%
# =========================
# CELDA 10) Ajuste final M1b-clean
# =========================

ALL_YEARS = [2020, 2021, 2022, 2023, 2024]

DRAWS_FINAL_C = 1200
TUNE_FINAL_C = 1800
CHAINS_FINAL_C = 4
TARGET_ACCEPT_FINAL_C = 0.99
MAX_TREEDEPTH_FINAL_C = 15
RANDOM_SEED_FINAL_C = 42

# -------------------------------------------------
# 1) Escalar covariables limpias con todo el período
# -------------------------------------------------
grid_scaled_all_c, scale_params_all_c = scale_grid_by_train_years(
    grid_df=grid_base_clean,
    x_cols=X_COLS_NO2_CLEAN,
    train_years=ALL_YEARS
)

obs_scaled_all_c = merge_scaled_covariates_to_obs(
    obs_df=obs_panel_clean,
    grid_scaled=grid_scaled_all_c,
    x_cols=X_COLS_NO2_CLEAN
)

z_cols_c = [f"{c}_z" for c in X_COLS_NO2_CLEAN]

# -------------------------------------------------
# 2) Arrays de entrenamiento completo
# -------------------------------------------------
X_all_c = obs_scaled_all_c[z_cols_c].to_numpy(dtype=float)
y_all_raw_c = obs_scaled_all_c["NO2_obs"].to_numpy(dtype=float)
y_all_c = np.log(y_all_raw_c + EPS) if USE_LOG else y_all_raw_c

cell_all_c = obs_scaled_all_c["cell_idx"].to_numpy(dtype=int)
time_all_c = obs_scaled_all_c["time_idx"].to_numpy(dtype=int)

ei = w_edges["i"].to_numpy(dtype=int)
ej = w_edges["j"].to_numpy(dtype=int)

p_c = X_all_c.shape[1]
alpha_mu_all_c = float(np.mean(y_all_c))

print("Ajustando modelo final M1b-clean...")
print("- observaciones válidas:", len(obs_scaled_all_c))
print("- estaciones únicas    :", obs_scaled_all_c["Estacion"].nunique())
print("- celdas observadas     :", obs_scaled_all_c["cell_id"].nunique())
print("- celdas totales malla  :", len(cell_ids))
print("- meses totales         :", len(time_ids))
print("- covariables usadas    :", X_COLS_NO2_CLEAN)

# -------------------------------------------------
# 3) Ajuste final M1b-clean
# -------------------------------------------------
with no.Model() as final_model_m1bclean:
    # efectos fijos regularizados
    alpha = no.Normal("alpha", mu=alpha_mu_all_c, sigma=1.0)
    beta = no.Normal("beta", mu=0.0, sigma=0.5, shape=p_c)

    # error observacional
    sigma_y = no.HalfNormal("sigma_y", sigma=0.75)

    # espacial ICAR
    tau_phi = no.Exponential("tau_phi", 2.0)
    phi_raw = no.Normal("phi_raw", mu=0.0, sigma=1.0, shape=len(cell_ids))
    phi = no.Deterministic("phi", phi_raw - pt.mean(phi_raw))
    no.Potential("icar_penalty", -0.5 * tau_phi * pt.sum((phi[ei] - phi[ej]) ** 2))

    # temporal RW1
    sigma_t = no.HalfNormal("sigma_t", sigma=0.25)
    delta_raw = no.GaussianRandomWalk("delta_raw", sigma=sigma_t, shape=len(time_ids))
    delta = no.Deterministic("delta", delta_raw - pt.mean(delta_raw))

    # media
    mu_all_c = alpha + pt.dot(X_all_c, beta) + phi[cell_all_c] + delta[time_all_c]

    # likelihood
    no.Normal("y_obs", mu=mu_all_c, sigma=sigma_y, observed=y_all_c)

    # muestreo
    idata_final_m1bclean = no.sample(
        draws=DRAWS_FINAL_C,
        tune=TUNE_FINAL_C,
        chains=CHAINS_FINAL_C,
        init="adapt_diag",
        target_accept=TARGET_ACCEPT_FINAL_C,
        max_treedepth=MAX_TREEDEPTH_FINAL_C,
        random_seed=RANDOM_SEED_FINAL_C,
        return_inferencedata=True,
        progressbar=True,
    )

# -------------------------------------------------
# 4) Predicción sobre toda la malla y todos los meses
# -------------------------------------------------
X_grid_all_c = grid_scaled_all_c[z_cols_c].to_numpy(dtype=float)
cell_grid_all_c = grid_scaled_all_c["cell_idx"].to_numpy(dtype=int)
time_grid_all_c = grid_scaled_all_c["time_idx"].to_numpy(dtype=int)

p05_grid_c, p50_grid_c, p95_grid_c = posterior_predict_concentration(
    idata=idata_final_m1bclean,
    X_mat=X_grid_all_c,
    cell_idx_arr=cell_grid_all_c,
    time_idx_arr=time_grid_all_c,
    eps=EPS
)

surface_final_m1bclean = grid_scaled_all_c[
    ["cell_id", "fecha", "year", "month", "cell_idx", "time_idx"]
].copy()

surface_final_m1bclean["p05_hbm"] = p05_grid_c
surface_final_m1bclean["p50_hbm"] = p50_grid_c
surface_final_m1bclean["p95_hbm"] = p95_grid_c
surface_final_m1bclean["width_90"] = (
    surface_final_m1bclean["p95_hbm"] - surface_final_m1bclean["p05_hbm"]
)

# -------------------------------------------------
# 5) Summary y diagnósticos finales
# -------------------------------------------------
summary_final_m1bclean = az.summary(
    idata_final_m1bclean,
    var_names=["alpha", "beta", "sigma_y", "tau_phi", "sigma_t"],
    round_to=4
)

diag_final_m1bclean = pd.DataFrame([{
    "model": "M1b_clean_final",
    "max_r_hat": float(summary_final_m1bclean["r_hat"].max()),
    "min_ess_bulk": float(summary_final_m1bclean["ess_bulk"].min()),
    "min_ess_tail": float(summary_final_m1bclean["ess_tail"].min()),
    "r_hat_ok": bool(summary_final_m1bclean["r_hat"].max() <= 1.01),
    "ess_bulk_ok": bool(summary_final_m1bclean["ess_bulk"].min() >= 400),
    "ess_tail_ok": bool(summary_final_m1bclean["ess_tail"].min() >= 400),
}])

# -------------------------------------------------
# 6) Guardar salidas
# -------------------------------------------------
surface_final_clean_path = OUT_DIR / "HBM_NO2_M1bclean_surface_final.csv"
summary_final_clean_path = OUT_DIR / "HBM_NO2_M1bclean_final_summary.csv"
diag_final_clean_path = OUT_DIR / "HBM_NO2_M1bclean_final_diagnostics.csv"
scale_final_clean_path = OUT_DIR / "HBM_NO2_M1bclean_final_scale_params.json"
idata_final_clean_path = OUT_DIR / "HBM_NO2_M1bclean_final_posterior.nc"

surface_final_m1bclean.to_csv(surface_final_clean_path, index=False)
summary_final_m1bclean.to_csv(summary_final_clean_path)
diag_final_m1bclean.to_csv(diag_final_clean_path, index=False)

with open(scale_final_clean_path, "w", encoding="utf-8") as f:
    json.dump(scale_params_all_c, f, ensure_ascii=False, indent=2)

az.to_netcdf(idata_final_m1bclean, idata_final_clean_path)

# -------------------------------------------------
# 7) Resumen final
# -------------------------------------------------
print("\nAJUSTE FINAL M1b-clean COMPLETADO")
print("- superficie final     :", surface_final_clean_path)
print("- summary final        :", summary_final_clean_path)
print("- diagnósticos finales :", diag_final_clean_path)
print("- scale params         :", scale_final_clean_path)
print("- posterior netcdf     :", idata_final_clean_path)

print("\nDiagnósticos finales:")
display(diag_final_m1bclean)

print("\nPrimeras filas de la superficie final limpia:")
display(surface_final_m1bclean.head())

Ajustando modelo final M1b-clean...
- observaciones válidas: 691
- estaciones únicas    : 15
- celdas observadas     : 13
- celdas totales malla  : 254
- meses totales         : 60
- covariables usadas    : ['Vel_viento_idw_clean', 'diff_NO2_grid_slog_clean', 'adv_proxy_NO2_grid_clean']


Initializing NUTS using adapt_diag...
ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: local_subtensor_merge
ERROR (pytensor.graph.rewriting.basic): node: Subtensor{i}(Subtensor{start:}.0, 0)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 1920, in process_node
    replacements = node_rewriter.transform(
                   ^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 993, in transform
    return self.fn(fgraph, node)
           ^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 416, in local_subtensor_merge
    merge_two_slices(
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 865, in

Output()

Sampling 4 chains for 1_800 tune and 1_200 draw iterations (7_200 + 4_800 draws total) took 562 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



AJUSTE FINAL M1b-clean COMPLETADO
- superficie final     : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1bclean_surface_final.csv
- summary final        : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1bclean_final_summary.csv
- diagnósticos finales : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1bclean_final_diagnostics.csv
- scale params         : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1bclean_final_scale_params.json
- posterior netcdf     : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1bclean_final_posterior.nc

Diagnósticos finales:


,model,max_r_hat,min_ess_bulk,min_ess_tail,r_hat_ok,ess_bulk_ok,ess_tail_ok
0,M1b_clean_final,1.0064,550.3392,1424.4628,True,True,True



Primeras filas de la superficie final limpia:


,cell_id,fecha,year,month,cell_idx,time_idx,p05_hbm,p50_hbm,p95_hbm,width_90
0,0,2020-01-01,2020,1,0,0,2.289092,12.366804,68.611466,66.322374
1,1,2020-01-01,2020,1,1,0,2.243222,12.694891,69.271697,67.028474
2,2,2020-01-01,2020,1,2,0,2.220887,12.186088,69.124685,66.903799
3,41,2020-01-01,2020,1,3,0,2.389146,12.595855,64.744632,62.355485
4,42,2020-01-01,2020,1,4,0,2.340853,11.955054,63.574310,61.233456



# CELDA 10B — Diagnóstico focal del summary final del modelo limpio

 **Objetivo:**
 identificar qué parámetro(s) del modelo final `M1b_clean` están
 generando el `max_r_hat` más alto y revisar si el problema es puntual
 o extendido.

 **Qué hace esta celda:**
 1. Lee `HBM_NO2_M1bclean_final_summary.csv`.
 2. Ordena el summary por `r_hat` de mayor a menor.
 3. Marca parámetros con:
    - `r_hat > 1.01`
    - `ess_bulk < 400`
    - `ess_tail < 400`
 4. Muestra:
    - top 20 parámetros con peor `r_hat`
    - subconjunto de parámetros problemáticos
 5. Guarda una tabla de diagnóstico focal.

 **Interpretación esperada:**
 - si solo 1 o pocos parámetros quedan apenas por encima de 1.01,
   el modelo sigue siendo razonablemente usable;
 - si son muchos, habría que volver a ajustar.

In [19]:
# %%
# =========================
# CELDA 10B) Diagnóstico focal del summary final limpio
# =========================

summary_final_clean_path = OUT_DIR / "HBM_NO2_M1bclean_final_summary.csv"
summary_final_clean = pd.read_csv(summary_final_clean_path, index_col=0)

needed_cols = ["mean", "sd", "ess_bulk", "ess_tail", "r_hat"]
missing_cols = [c for c in needed_cols if c not in summary_final_clean.columns]
if missing_cols:
    raise ValueError(f"Faltan columnas en el summary final: {missing_cols}")

summary_diag = summary_final_clean.copy().reset_index().rename(columns={"index": "parametro"})

summary_diag["flag_rhat"] = summary_diag["r_hat"] > 1.01
summary_diag["flag_ess_bulk"] = summary_diag["ess_bulk"] < 400
summary_diag["flag_ess_tail"] = summary_diag["ess_tail"] < 400

summary_diag["n_flags"] = (
    summary_diag["flag_rhat"].astype(int) +
    summary_diag["flag_ess_bulk"].astype(int) +
    summary_diag["flag_ess_tail"].astype(int)
)

# top por rhat
top_rhat = summary_diag.sort_values(["r_hat", "ess_bulk"], ascending=[False, True]).head(20).copy()

# parámetros problemáticos
problem_params = summary_diag.loc[
    (summary_diag["flag_rhat"]) |
    (summary_diag["flag_ess_bulk"]) |
    (summary_diag["flag_ess_tail"])
].sort_values(["n_flags", "r_hat", "ess_bulk"], ascending=[False, False, True]).copy()

# guardar
summary_diag_path = OUT_DIR / "HBM_NO2_M1bclean_final_summary_diagnostic_focus.csv"
summary_diag.to_csv(summary_diag_path, index=False)

print("Archivo guardado:")
print("-", summary_diag_path)

print("\nResumen global del summary final:")
print("- número total de parámetros        :", len(summary_diag))
print("- parámetros con r_hat > 1.01       :", int(summary_diag["flag_rhat"].sum()))
print("- parámetros con ess_bulk < 400     :", int(summary_diag["flag_ess_bulk"].sum()))
print("- parámetros con ess_tail < 400     :", int(summary_diag["flag_ess_tail"].sum()))

print("\nTop 20 parámetros con mayor r_hat:")
display(top_rhat[["parametro", "mean", "sd", "ess_bulk", "ess_tail", "r_hat", "n_flags"]])

print("\nParámetros problemáticos:")
display(problem_params[["parametro", "mean", "sd", "ess_bulk", "ess_tail", "r_hat", "flag_rhat", "flag_ess_bulk", "flag_ess_tail"]])

Archivo guardado:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1bclean_final_summary_diagnostic_focus.csv

Resumen global del summary final:
- número total de parámetros        : 7
- parámetros con r_hat > 1.01       : 0
- parámetros con ess_bulk < 400     : 0
- parámetros con ess_tail < 400     : 0

Top 20 parámetros con mayor r_hat:


,parametro,mean,sd,ess_bulk,ess_tail,r_hat,n_flags
0,alpha,2.6448,0.2633,550.3392,1424.4628,1.0064,0
5,tau_phi,0.0013,0.0013,4999.8425,2314.3312,1.0027,0
1,beta[0],-0.0817,0.0213,7941.7190,3529.7749,1.0020,0
4,sigma_y,0.2055,0.0058,6506.4366,3613.5153,1.0009,0
6,sigma_t,0.1344,0.0170,5629.4978,3446.3704,1.0007,0
2,beta[1],-0.0306,0.0073,8351.4908,3150.8567,1.0007,0
3,beta[2],0.0008,0.0091,7694.6705,3583.8814,1.0000,0



Parámetros problemáticos:


,parametro,mean,sd,ess_bulk,ess_tail,r_hat,flag_rhat,flag_ess_bulk,flag_ess_tail
